# The Carbon Market MRV Problem and Our Solution

Global warming is accelerating due to excess carbon emissions. To achieve **Net Zero**, countries and corporations are mandated to purchase carbon credits generated by projects that sequester carbon, such as planting mangroves (Blue Carbon). However, verifying exactly how much carbon these projects sequester is currently a massive bottleneck. It requires expensive manual teams on the ground to measure and report.

**Our solution automates MRV (Measurement, Reporting, and Verification)**. By leveraging satellite imagery and spatio-temporal deep learning, we eliminate the need for manual teams. While this notebook focuses on Mangroves (Blue Carbon), this architecture is built to scale across all forms of carbon (Green, White, etc.) globally.

# Gather S1 Satelite Data


In [ ]:
print("Installing geemap...")
!pip install geemap

Installing geemap...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 324, in run
    session = self.get_default_session(options)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/index_command.py", line 71, in get_default_session
    self._session = self.enter_context(self._build_session(options))
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/index_command.py", line 100, in _build_session
    session = PipSession(
  

In [ ]:
print("Installing earthengine-api...")
!pip install earthengine-api

Installing earthengine-api...


In [ ]:
!pip install earthaccess

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 36.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [ ]:
import pandas as pd

csv_path = '/content/drive/MyDrive/STAGE 1/S2_UAE_MANGROVE.CSV'
df_uae = pd.read_csv(csv_path)

print(f"Successfully loaded CSV from: {csv_path}")
print("DataFrame head:")
print(df_uae.head())

Successfully loaded CSV from: /content/drive/MyDrive/STAGE 1/S2_UAE_MANGROVE.CSV
DataFrame head:
    latitude  longitude  B01  B02     B03  B04     B05  B06  B07     B08  ...  \
0  25.958120  55.103933  0.0  0.0  0.0000  0.0  0.0000  0.0  0.0  0.0001  ...   
1  25.956220  55.099749  0.0  0.0  0.0000  0.0  0.0000  0.0  0.0  0.0002  ...   
2  25.956220  55.101841  0.0  0.0  0.0000  0.0  0.0000  0.0  0.0  0.0007  ...   
3  25.956220  55.103933  0.0  0.0  0.0000  0.0  0.0000  0.0  0.0  0.0000  ...   
4  25.954321  55.095566  0.0  0.0  0.0016  0.0  0.0002  0.0  0.0  0.0004  ...   

   False_color_(urban)  Highlight_Optimized_Natural_Color_  Moisture_index  \
0              0.02425                                 0.0        0.501961   
1              0.02400                                 0.0        0.501961   
2              0.02475                                 0.0        0.619608   
3              0.02475                                 0.0        0.501961   
4              0.02475    

In [ ]:
df_uae.columns

Index(['latitude', 'longitude', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06',
       'B07', 'B08', 'B8A', 'B09', 'B11', 'B12', 'False_color',
       'False_color_(urban)', 'Highlight_Optimized_Natural_Color_',
       'Moisture_index', 'NDSI', 'NDVI', 'NDWI', 'Scene_classification_map',
       'SWIR', 'True_color', 'category'],
      dtype='object')

In [ ]:
df_uae.category.value_counts()

,count
category,
Urban,1322795
Sand,970907
Water,642282
Forest,98863
Mangrove,4010


In [ ]:
# Filter the DataFrame based on category and coordinates
df_uae_mangrove_filtered = df_uae[df_uae['category'] == 'Mangrove']

print("Filtered DataFrame head:")
print(df_uae_mangrove_filtered.head())
print(f"\nShape of filtered DataFrame: {df_uae_mangrove_filtered.shape}")

Filtered DataFrame head:
         latitude  longitude     B01     B02     B03     B04     B05     B06  \
9925    25.739637  54.859193  0.0347  0.0200  0.0237  0.0206  0.0209  0.0196   
81118   25.346367  53.574829  0.0534  0.0484  0.0517  0.0530  0.0626  0.0606   
87838   25.329268  54.673023  0.0387  0.0394  0.0518  0.0540  0.0578  0.0510   
118866  25.257074  54.744144  0.0388  0.0697  0.0718  0.0729  0.0766  0.0741   
126391  25.241875  54.756695  0.0569  0.0568  0.0573  0.0623  0.0646  0.0674   

           B07     B08  ...  False_color_(urban)  \
9925    0.0184  0.1728  ...              0.03875   
81118   0.0601  0.0043  ...              0.10400   
87838   0.0491  0.0051  ...              0.03400   
118866  0.0700  0.0022  ...              0.07450   
126391  0.0817  0.2171  ...              0.08225   

        Highlight_Optimized_Natural_Color_  Moisture_index     NDSI      NDVI  \
9925                              0.058258             0.0  0.05150  0.176471   
81118              

In [ ]:
import ee

ee.Authenticate()

ee.Initialize(project='mangroove-startup')

print("Earth Engine authenticated and initialized successfully for project 'mangroove-startup'.")

Earth Engine authenticated and initialized successfully for project 'mangroove-startup'.


In [ ]:
mangrove_coords_uae = df_uae_mangrove_filtered[['latitude', 'longitude']]

mangrove_ee_features_uae = []

for index, row in mangrove_coords_uae.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    mangrove_ee_features_uae.append(ee.Feature(point))

mangrove_feature_collection_uae = ee.FeatureCollection(mangrove_ee_features_uae)

print(f"Size of the Earth Engine FeatureCollection for uae mangroves: {mangrove_feature_collection_uae.size().getInfo()}")
print("Conversion to Earth Engine FeatureCollection successful for uae mangroves.")

Size of the Earth Engine FeatureCollection for uae mangroves: 4010
Conversion to Earth Engine FeatureCollection successful for uae mangroves.


In [ ]:
start_date = '2021-01-01'
end_date = '2021-12-31'

# Load Sentinel-1 GRD ImageCollection for backscatter
sentinel1_backscatter_uae = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterDate(start_date, end_date) \
    .filterBounds(mangrove_feature_collection_uae) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
    .filter(ee.Filter.eq('productType', 'GRD'))

print(f"Sentinel-1 backscatter collection size for uae: {sentinel1_backscatter_uae.size().getInfo()}")
print("Sentinel-1 backscatter data loaded and filtered successfully for uae.")

Sentinel-1 backscatter collection size for uae: 273
Sentinel-1 backscatter data loaded and filtered successfully for uae.


In [ ]:
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

sentinel1_backscatter_uae_filtered = sentinel1_backscatter_uae.map(apply_speckle_filter)

median_backscatter_uae = sentinel1_backscatter_uae_filtered.median().select(['VV', 'VH'])

print("Speckle filtering applied and median backscatter composite created for VV and VH bands for uae.")

Speckle filtering applied and median backscatter composite created for VV and VH bands for uae.


In [ ]:
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    print(f"Processing {total_features} features in chunks of {chunk_size}...")

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        # Slice the FeatureCollection for the current chunk
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)

        # Convert the List to a FeatureCollection for reduceRegions
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        # Apply reduceRegions for the current chunk
        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        # Fetch the results for the current chunk
        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
            print(f"Fetched chunk {i}-{end_index}: {len(chunk_list)} features.")
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            # Depending on the error, you might want to re-try or skip
            continue
    return all_extracted_features

# Extract VV and VH backscatter values for uae mangroves
backscatter_features_uae = fetch_features_in_chunks(
    mangrove_feature_collection_uae,
    median_backscatter_uae
)

print(f"Extracted {len(backscatter_features_uae)} backscatter values for uae mangroves.")

Processing 4010 features in chunks of 250...
Fetched chunk 0-250: 250 features.
Fetched chunk 250-500: 250 features.
Fetched chunk 500-750: 250 features.
Fetched chunk 750-1000: 250 features.
Fetched chunk 1000-1250: 250 features.
Fetched chunk 1250-1500: 250 features.
Fetched chunk 1500-1750: 250 features.
Fetched chunk 1750-2000: 250 features.
Fetched chunk 2000-2250: 250 features.
Fetched chunk 2250-2500: 250 features.
Fetched chunk 2500-2750: 250 features.
Fetched chunk 2750-3000: 250 features.
Fetched chunk 3000-3250: 250 features.
Fetched chunk 3250-3500: 250 features.
Fetched chunk 3500-3750: 250 features.
Fetched chunk 3750-4000: 250 features.
Fetched chunk 4000-4010: 10 features.
Extracted 4010 backscatter values for uae mangroves.


In [ ]:
start_date = '2021-01-01'
end_date = '2021-12-31'

# Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
sentinel1_coherence_uae = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
    .filterDate(start_date, end_date) \
    .filterBounds(mangrove_feature_collection_uae) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

# Apply speckle filter to the coherence collection
sentinel1_coherence_uae_filtered = sentinel1_coherence_uae.map(apply_speckle_filter)

# Create a median composite image from the filtered coherence collection, selecting VV and VH bands
median_coherence_uae = sentinel1_coherence_uae_filtered.median().select(['VV', 'VH'])

print(f"Sentinel-1 coherence collection size for uae: {sentinel1_coherence_uae.size().getInfo()}")
print("Sentinel-1 coherence data loaded, filtered, and processed successfully for uae.")

Sentinel-1 coherence collection size for uae: 273
Sentinel-1 coherence data loaded, filtered, and processed successfully for uae.


In [ ]:
 coherence_features_uae = fetch_features_in_chunks(
    mangrove_feature_collection_uae,
    median_coherence_uae
)

print(f"Extracted {len(coherence_features_uae)} coherence values for uae mangroves.")

Processing 4010 features in chunks of 250...
Fetched chunk 0-250: 250 features.
Fetched chunk 250-500: 250 features.
Fetched chunk 500-750: 250 features.
Fetched chunk 750-1000: 250 features.
Fetched chunk 1000-1250: 250 features.
Fetched chunk 1250-1500: 250 features.
Fetched chunk 1500-1750: 250 features.
Fetched chunk 1750-2000: 250 features.
Fetched chunk 2000-2250: 250 features.
Fetched chunk 2250-2500: 250 features.
Fetched chunk 2500-2750: 250 features.
Fetched chunk 2750-3000: 250 features.
Fetched chunk 3000-3250: 250 features.
Fetched chunk 3250-3500: 250 features.
Fetched chunk 3500-3750: 250 features.


Fetched chunk 3750-4000: 250 features.
Fetched chunk 4000-4010: 10 features.
Extracted 4010 coherence values for uae mangroves.


In [ ]:
backscatter_data_uae = []
for feature in backscatter_features_uae:
    properties = feature['properties']
    backscatter_data_uae.append({
        'VV_backscatter': properties.get('VV'),
        'VH_backscatter': properties.get('VH')
    })

coherence_data_uae = []
for feature in coherence_features_uae:
    properties = feature['properties']
    coherence_data_uae.append({
        'VV_coherence': properties.get('VV'),
        'VH_coherence': properties.get('VH')
    })

# Convert to DataFrame for easier merging
df_backscatter_uae = pd.DataFrame(backscatter_data_uae)
df_coherence_uae = pd.DataFrame(coherence_data_uae)

# Reset index of df_uae_mangrove_filtered to ensure proper concatenation
df_uae_mangrove_filtered_reset = df_uae_mangrove_filtered.reset_index(drop=True)

# Concatenate the new features with the uae mangrove DataFrame
df_uae_mangrove_updated = pd.concat([
    df_uae_mangrove_filtered_reset,
    df_backscatter_uae,
    df_coherence_uae
], axis=1)

print("Successfully integrated Sentinel-1 backscatter and coherence features into df_uae_mangrove_filtered.")
print("DataFrame head with new features:")
print(df_uae_mangrove_updated.head())

Successfully integrated Sentinel-1 backscatter and coherence features into df_uae_mangrove_filtered.
DataFrame head with new features:
    latitude  longitude     B01     B02     B03     B04     B05     B06  \
0  25.739637  54.859193  0.0347  0.0200  0.0237  0.0206  0.0209  0.0196   
1  25.346367  53.574829  0.0534  0.0484  0.0517  0.0530  0.0626  0.0606   
2  25.329268  54.673023  0.0387  0.0394  0.0518  0.0540  0.0578  0.0510   
3  25.257074  54.744144  0.0388  0.0697  0.0718  0.0729  0.0766  0.0741   
4  25.241875  54.756695  0.0569  0.0568  0.0573  0.0623  0.0646  0.0674   

      B07     B08  ...      NDVI      NDWI  Scene_classification_map     SWIR  \
0  0.0184  0.1728  ...  0.176471  0.050980                       0.0  0.03875   
1  0.0601  0.0043  ...  0.254902  0.000000                       0.0  0.10400   
2  0.0491  0.0051  ...  0.286275  0.000000                       0.0  0.03400   
3  0.0700  0.0022  ...  0.125490  0.000000                       0.0  0.07450   
4  0.0817

In [ ]:
print(df_uae_mangrove_updated.VV_backscatter.describe())
print(df_uae_mangrove_updated.VH_backscatter.describe())
print(df_uae_mangrove_updated.VV_coherence.describe())
print(df_uae_mangrove_updated.VH_coherence.describe())

count    4010.000000
mean      -11.469964
std         4.848585
min       -25.197835
25%       -13.622791
50%        -9.408340
75%        -8.130661
max         7.045174
Name: VV_backscatter, dtype: float64
count    4010.000000
mean      -18.880417
std         5.128193
min       -32.348410
25%       -22.535091
50%       -16.516018
75%       -15.460192
max        -1.998425
Name: VH_backscatter, dtype: float64
count    4010.000000
mean        0.107999
std         0.106885
min         0.003028
25%         0.043431
50%         0.114596
75%         0.153796
max         5.065222
Name: VV_coherence, dtype: float64
count    4010.000000
mean        0.020961
std         0.019599
min         0.000582
25%         0.005582
50%         0.022305
75%         0.028467
max         0.631238
Name: VH_coherence, dtype: float64


In [ ]:
output_csv_path_uae = '/content/drive/MyDrive/STAGE 1/S1_S2_UAE.csv'
df_uae_mangrove_updated.to_csv(output_csv_path_uae, index=False)
print(f"Final combined mangrove data for uae saved to: {output_csv_path_uae}")

Final combined mangrove data for uae saved to: /content/drive/MyDrive/STAGE 1/S1_S2_UAE.csv


# S1 UAE Monthly Data Extraction

In [ ]:
import pandas as pd

df=pd.read_csv('/content/drive/MyDrive/STAGE 1/S2_UAE_MANGROVE.CSV')
df=df[df['category'] == 'Mangrove'][['latitude', 'longitude']].reset_index(drop=True)
df.head()

,latitude,longitude
0,25.739637,54.859193
1,25.346367,53.574829
2,25.329268,54.673023
3,25.257074,54.744144
4,25.241875,54.756695


In [ ]:
import pandas as pd

start_date = '2021-01-01'
end_date = '2026-01-31'

# Generate monthly dates
monthly_dates = pd.date_range(start=start_date, end=end_date, freq='MS').tolist()

print(f"Generated {len(monthly_dates)} monthly dates:")
print(monthly_dates)


Generated 61 monthly dates:
[Timestamp('2021-01-01 00:00:00'), Timestamp('2021-02-01 00:00:00'), Timestamp('2021-03-01 00:00:00'), Timestamp('2021-04-01 00:00:00'), Timestamp('2021-05-01 00:00:00'), Timestamp('2021-06-01 00:00:00'), Timestamp('2021-07-01 00:00:00'), Timestamp('2021-08-01 00:00:00'), Timestamp('2021-09-01 00:00:00'), Timestamp('2021-10-01 00:00:00'), Timestamp('2021-11-01 00:00:00'), Timestamp('2021-12-01 00:00:00'), Timestamp('2022-01-01 00:00:00'), Timestamp('2022-02-01 00:00:00'), Timestamp('2022-03-01 00:00:00'), Timestamp('2022-04-01 00:00:00'), Timestamp('2022-05-01 00:00:00'), Timestamp('2022-06-01 00:00:00'), Timestamp('2022-07-01 00:00:00'), Timestamp('2022-08-01 00:00:00'), Timestamp('2022-09-01 00:00:00'), Timestamp('2022-10-01 00:00:00'), Timestamp('2022-11-01 00:00:00'), Timestamp('2022-12-01 00:00:00'), Timestamp('2023-01-01 00:00:00'), Timestamp('2023-02-01 00:00:00'), Timestamp('2023-03-01 00:00:00'), Timestamp('2023-04-01 00:00:00'), Timestamp('2023-05-

In [ ]:
import ee
import pandas as pd

# Ensure ee is initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the speckle filter function (if not already defined in the current session)
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# Define the fetch_features_in_chunks function (if not already defined in the current session)
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    # print(f"Processing {total_features} features in chunks of {chunk_size}...") # Commented out for cleaner output in loop

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        # Slice the FeatureCollection for the current chunk
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)

        # Convert the List to a FeatureCollection for reduceRegions
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        # Apply reduceRegions for the current chunk
        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        # Fetch the results for the current chunk
        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
            # print(f"Fetched chunk {i}-{end_index}: {len(chunk_list)} features.") # Commented out for cleaner output in loop
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            # Depending on the error, you might want to re-try or skip
            continue
    return all_extracted_features

# Convert the df (mangrove coordinates) to an Earth Engine FeatureCollection
mangrove_ee_features = []
for index, row in df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    mangrove_ee_features.append(ee.Feature(point))
mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

all_monthly_data = []

print(f"Starting monthly data extraction for {len(monthly_dates)} periods...")

for i in range(len(monthly_dates) - 1):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    all_monthly_data.append(monthly_combined_df)

# Concatenate all monthly DataFrames into a single DataFrame
final_df = pd.concat(all_monthly_data, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_monthly = '/content/drive/MyDrive/STAGE 1/S1_S2_UAE_Monthly.csv'
final_df.to_csv(output_csv_path_monthly, index=False)

print("Monthly data extraction complete.")
print(f"Final combined monthly data saved to: {output_csv_path_monthly}")
print("Final DataFrame head:")
print(final_df.head())
print(f"Final DataFrame shape: {final_df.shape}")

Starting monthly data extraction for 61 periods...
Processing data for month: 2021-01-01 to 2021-02-01
Processing data for month: 2021-02-01 to 2021-03-01
Processing data for month: 2021-03-01 to 2021-04-01
Processing data for month: 2021-04-01 to 2021-05-01
Processing data for month: 2021-05-01 to 2021-06-01
Processing data for month: 2021-06-01 to 2021-07-01
Processing data for month: 2021-07-01 to 2021-08-01
Processing data for month: 2021-08-01 to 2021-09-01
Processing data for month: 2021-09-01 to 2021-10-01
Processing data for month: 2021-10-01 to 2021-11-01
Processing data for month: 2021-11-01 to 2021-12-01
Processing data for month: 2021-12-01 to 2022-01-01
Processing data for month: 2022-01-01 to 2022-02-01


KeyboardInterrupt: 

In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame
mangrove_ee_features = []
for index, row in df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    mangrove_ee_features.append(ee.Feature(point))
mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available (it should be from previous execution)
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices
chunk_start_idx = 12
chunk_size = 6
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates))

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop from chunk_start_idx to chunk_end_idx - 1
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    all_monthly_data.append(monthly_combined_df)

print("Chunk data extraction complete.")

Continuing monthly data extraction for periods from index 12 to 17...
Processing data for month: 2022-01-01 to 2022-02-01
Processing data for month: 2022-02-01 to 2022-03-01
Processing data for month: 2022-03-01 to 2022-04-01
Processing data for month: 2022-04-01 to 2022-05-01
Processing data for month: 2022-05-01 to 2022-06-01
Processing data for month: 2022-06-01 to 2022-07-01
Chunk data extraction complete.


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices
chunk_start_idx = 18
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) -1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")


Continuing monthly data extraction for periods from index 18 to 23...
Processing data for month: 2022-07-01 to 2022-08-01
Processing data for month: 2022-08-01 to 2022-09-01
Processing data for month: 2022-09-01 to 2022-10-01
Processing data for month: 2022-10-01 to 2022-11-01
Processing data for month: 2022-11-01 to 2022-12-01
Processing data for month: 2022-12-01 to 2023-01-01


Finished processing months from index 18 to 23.
Total months processed so far: 24


**Reasoning**:
The previous execution successfully processed months up to index 23. To continue the iterative data extraction, I will now define the next chunk of months starting from index 24, and process them following the same logic, appending the extracted backscatter and coherence data to the `all_monthly_data` list.



In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 23. So, start from 24.
chunk_start_idx = 24
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")


Continuing monthly data extraction for periods from index 24 to 29...
Processing data for month: 2023-01-01 to 2023-02-01
Processing data for month: 2023-02-01 to 2023-03-01
Processing data for month: 2023-03-01 to 2023-04-01
Processing data for month: 2023-04-01 to 2023-05-01
Processing data for month: 2023-05-01 to 2023-06-01
Processing data for month: 2023-06-01 to 2023-07-01
Finished processing months from index 24 to 29.
Total months processed so far: 30


**Reasoning**:
The previous execution successfully processed months up to index 29 (total 30 months). To continue the iterative data extraction, I will define the next chunk of months starting from index 30 (July 2023) and process them following the same logic, appending the extracted backscatter and coherence data to the `all_monthly_data` list.



In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 29. So, start from 30.
chunk_start_idx = 30
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")


Continuing monthly data extraction for periods from index 30 to 35...
Processing data for month: 2023-07-01 to 2023-08-01
Processing data for month: 2023-08-01 to 2023-09-01
Processing data for month: 2023-09-01 to 2023-10-01
Processing data for month: 2023-10-01 to 2023-11-01
Processing data for month: 2023-11-01 to 2023-12-01


Processing data for month: 2023-12-01 to 2024-01-01
Finished processing months from index 30 to 35.
Total months processed so far: 36


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 35. So, start from 36.
chunk_start_idx = 36
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")

Continuing monthly data extraction for periods from index 36 to 41...
Processing data for month: 2024-01-01 to 2024-02-01
Processing data for month: 2024-02-01 to 2024-03-01
Processing data for month: 2024-03-01 to 2024-04-01
Processing data for month: 2024-04-01 to 2024-05-01
Processing data for month: 2024-05-01 to 2024-06-01
Processing data for month: 2024-06-01 to 2024-07-01
Finished processing months from index 36 to 41.
Total months processed so far: 42


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 41. So, start from 42.
chunk_start_idx = 42
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")

Continuing monthly data extraction for periods from index 42 to 47...
Processing data for month: 2024-07-01 to 2024-08-01
Processing data for month: 2024-08-01 to 2024-09-01
Processing data for month: 2024-09-01 to 2024-10-01
Processing data for month: 2024-10-01 to 2024-11-01


Processing data for month: 2024-11-01 to 2024-12-01
Processing data for month: 2024-12-01 to 2025-01-01
Finished processing months from index 42 to 47.
Total months processed so far: 48


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 47. So, start from 48.
chunk_start_idx = 48
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")


Continuing monthly data extraction for periods from index 48 to 53...
Processing data for month: 2025-01-01 to 2025-02-01
Processing data for month: 2025-02-01 to 2025-03-01
Processing data for month: 2025-03-01 to 2025-04-01
Processing data for month: 2025-04-01 to 2025-05-01
Processing data for month: 2025-05-01 to 2025-06-01
Processing data for month: 2025-06-01 to 2025-07-01
Finished processing months from index 48 to 53.
Total months processed so far: 54


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 53. So, start from 54.
chunk_start_idx = 54
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")


Continuing monthly data extraction for periods from index 54 to 59...
Processing data for month: 2025-07-01 to 2025-08-01
Processing data for month: 2025-08-01 to 2025-09-01
Processing data for month: 2025-09-01 to 2025-10-01


Processing data for month: 2025-10-01 to 2025-11-01
Processing data for month: 2025-11-01 to 2025-12-01
Processing data for month: 2025-12-01 to 2026-01-01
Finished processing months from index 54 to 59.
Total months processed so far: 60


In [ ]:
import pandas as pd

# Concatenate all monthly DataFrames into a single DataFrame
final_df = pd.concat(all_monthly_data, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_monthly = '/content/drive/MyDrive/STAGE 1/S1_S2_UAE_Monthly.csv'
final_df.to_csv(output_csv_path_monthly, index=False)

print("Monthly data extraction complete.")
print(f"Final combined monthly data saved to: {output_csv_path_monthly}")
print("Final DataFrame head:")
print(final_df.head())
print(f"Final DataFrame shape: {final_df.shape}")

Monthly data extraction complete.
Final combined monthly data saved to: /content/drive/MyDrive/STAGE 1/S1_S2_UAE_Monthly.csv
Final DataFrame head:
    latitude  longitude  VV_backscatter  VH_backscatter  VV_coherence  \
0  25.739637  54.859193      -20.039748      -28.833538      0.014012   
1  25.346367  53.574829      -23.080467      -29.506814      0.004920   
2  25.329268  54.673023      -19.840784      -27.015034      0.010576   
3  25.257074  54.744144      -17.796795      -26.082576      0.017216   
4  25.241875  54.756695      -17.313404      -26.774186      0.019055   

   VH_coherence       date  
0      0.001331 2021-01-01  
1      0.001120 2021-01-01  
2      0.002028 2021-01-01  
3      0.002494 2021-01-01  
4      0.002102 2021-01-01  
Final DataFrame shape: (240600, 7)


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the final month
# The previous execution processed up to index 59. So, start from 60.
chunk_start_idx = 60
chunk_size = 1
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
# For the last month, we need to handle monthly_dates[i+1] carefully if it goes out of bounds.
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates))

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")


Continuing monthly data extraction for periods from index 60 to 60...
Processing data for month: 2026-01-01 to 2026-02-01
Finished processing months from index 60 to 60.
Total months processed so far: 61


In [ ]:
import pandas as pd

# Concatenate all monthly DataFrames into a single DataFrame
final_df = pd.concat(all_monthly_data, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_monthly = '/content/drive/MyDrive/STAGE 1/S1_UAE_Monthly.csv'
final_df.to_csv(output_csv_path_monthly, index=False)

print("Monthly data extraction complete.")
print(f"Final combined monthly data saved to: {output_csv_path_monthly}")
print("Final DataFrame head:")
print(final_df.head())
print(f"Final DataFrame shape: {final_df.shape}")

NameError: name 'all_monthly_data' is not defined

In [ ]:
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/STAGE 1/S1_UAE_Monthly.csv')
df.shape

(244610, 7)

In [ ]:
df.to_csv('/content/drive/MyDrive/STAGE 1/S1_UAE_Monthly.csv')

# S2 UAE Monthly Data Extraction

In [ ]:
import ee
import pandas as pd

# 1. Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='mangroove-startup')
print("Earth Engine authenticated and initialized successfully for project 'mangroove-startup'.")

# 3. Reload the `df` DataFrame
csv_path = '/content/drive/MyDrive/STAGE 1/S2_UAE_MANGROVE.CSV'
df = pd.read_csv(csv_path)
df = df[df['category'] == 'Mangrove'][['latitude', 'longitude']].reset_index(drop=True)
print("Mangrove coordinates DataFrame 'df' reloaded and filtered.")

# 4. Regenerate the `monthly_dates` list
start_date = '2021-01-01'
end_date = '2026-01-31'
monthly_dates = pd.date_range(start=start_date, end=end_date, freq='MS').tolist()
print("Monthly dates list 'monthly_dates' regenerated.")

# 5. Convert the df DataFrame to an Earth Engine FeatureCollection
mangrove_ee_features = []
for index, row in df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    mangrove_ee_features.append(ee.Feature(point))
mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)
print(f"Earth Engine FeatureCollection created with {mangrove_feature_collection.size().getInfo()} features.")

# 6. Define a function add_derived_bands(image) to compute and add specific derived bands
def add_derived_bands(image):
    # Compute NDVI
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    # Compute NDWI (using B3 and B8, similar to original notebook's approach for NDWI with green and NIR)
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    # Compute NDSI (using B3 and B11 for green and SWIR1)
    ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')
    # Compute Moisture_index (NDMI using B8A and B11 for NIR and SWIR1)
    moisture_index = image.normalizedDifference(['B8A', 'B11']).rename('Moisture_index')

    # Select and rename other bands with valid names
    swir = image.select('B11').rename('SWIR')
    scl_map = image.select('SCL').rename('Scene_classification_map')

    # Corrected names for False_color_(urban) and Highlight_Optimized_Natural_Color_
    false_color = image.select('B8').rename('False_color') # Example band, can be customized
    false_color_urban = image.select('B12').rename('False_color_urban') # Using a valid name
    highlight_optimized_natural_color = image.select('B4').rename('Highlight_Optimized_Natural_Color') # Using a valid name
    true_color = image.select('B4').rename('True_color') # Example band, can be customized

    return image.addBands([
        ndvi, ndwi, ndsi, moisture_index, swir, scl_map,
        false_color, false_color_urban, highlight_optimized_natural_color, true_color
    ])
print("Utility function 'add_derived_bands' defined with corrected band names.")


# 7. Redefine the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features
print("Utility function 'fetch_features_in_chunks' redefined.")

# 8. Initialize an empty Python list for Sentinel-2 data
all_monthly_s2_data = []
print("'all_monthly_s2_data' list initialized.")

# 9. Define Sentinel-2 bands and derived indices to be extracted with corrected names
s2_all_bands_to_extract = [
    'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'SCL',
    'NDVI', 'NDWI', 'NDSI', 'Moisture_index', 'SWIR', 'Scene_classification_map',
    'False_color', 'False_color_urban', 'Highlight_Optimized_Natural_Color', 'True_color'
]
print(f"Sentinel-2 bands and indices defined with corrected names: {s2_all_bands_to_extract}")

Earth Engine authenticated and initialized successfully for project 'mangroove-startup'.
Mangrove coordinates DataFrame 'df' reloaded and filtered.
Monthly dates list 'monthly_dates' regenerated.
Earth Engine FeatureCollection created with 4010 features.
Utility function 'add_derived_bands' defined with corrected band names.
Utility function 'fetch_features_in_chunks' redefined.
'all_monthly_s2_data' list initialized.
Sentinel-2 bands and indices defined with corrected names: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'SCL', 'NDVI', 'NDWI', 'NDSI', 'Moisture_index', 'SWIR', 'Scene_classification_map', 'False_color', 'False_color_urban', 'Highlight_Optimized_Natural_Color', 'True_color']


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (January 2021 to June 2021, indices 0-5)
chunk_start_idx = 0
chunk_end_idx = 6

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 0 to 5...
Processing Sentinel-2 data for month: 2021-01-01 to 2021-02-01
Processing Sentinel-2 data for month: 2021-02-01 to 2021-03-01
Processing Sentinel-2 data for month: 2021-03-01 to 2021-04-01
Processing Sentinel-2 data for month: 2021-04-01 to 2021-05-01
Processing Sentinel-2 data for month: 2021-05-01 to 2021-06-01
Processing Sentinel-2 data for month: 2021-06-01 to 2021-07-01
Finished Sentinel-2 data extraction for months from index 0 to 5.
Total monthly Sentinel-2 DataFrames collected so far: 6


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (July 2021 to December 2021, indices 6-11)
chunk_start_idx = 6
chunk_end_idx = 12

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 6 to 11...
Processing Sentinel-2 data for month: 2021-07-01 to 2021-08-01
Processing Sentinel-2 data for month: 2021-08-01 to 2021-09-01
Processing Sentinel-2 data for month: 2021-09-01 to 2021-10-01
Processing Sentinel-2 data for month: 2021-10-01 to 2021-11-01
Processing Sentinel-2 data for month: 2021-11-01 to 2021-12-01


Processing Sentinel-2 data for month: 2021-12-01 to 2022-01-01
Finished Sentinel-2 data extraction for months from index 6 to 11.
Total monthly Sentinel-2 DataFrames collected so far: 61


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (January 2022 to June 2022, indices 12-17)
chunk_start_idx = 12
chunk_end_idx = 18

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 12 to 17...
Processing Sentinel-2 data for month: 2022-01-01 to 2022-02-01
Processing Sentinel-2 data for month: 2022-02-01 to 2022-03-01
Processing Sentinel-2 data for month: 2022-03-01 to 2022-04-01
Processing Sentinel-2 data for month: 2022-04-01 to 2022-05-01
Processing Sentinel-2 data for month: 2022-05-01 to 2022-06-01
Processing Sentinel-2 data for month: 2022-06-01 to 2022-07-01
Finished Sentinel-2 data extraction for months from index 12 to 17.
Total monthly Sentinel-2 DataFrames collected so far: 18


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (July 2022 to December 2022, indices 18-23)
chunk_start_idx = 18
chunk_end_idx = 24

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 18 to 23...
Processing Sentinel-2 data for month: 2022-07-01 to 2022-08-01
Processing Sentinel-2 data for month: 2022-08-01 to 2022-09-01


Processing Sentinel-2 data for month: 2022-09-01 to 2022-10-01
Processing Sentinel-2 data for month: 2022-10-01 to 2022-11-01
Processing Sentinel-2 data for month: 2022-11-01 to 2022-12-01
Processing Sentinel-2 data for month: 2022-12-01 to 2023-01-01
Finished Sentinel-2 data extraction for months from index 18 to 23.
Total monthly Sentinel-2 DataFrames collected so far: 24


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (January 2023 to June 2023, indices 24-29)
chunk_start_idx = 24
chunk_end_idx = 30

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 24 to 29...
Processing Sentinel-2 data for month: 2023-01-01 to 2023-02-01
Processing Sentinel-2 data for month: 2023-02-01 to 2023-03-01
Processing Sentinel-2 data for month: 2023-03-01 to 2023-04-01
Processing Sentinel-2 data for month: 2023-04-01 to 2023-05-01
Processing Sentinel-2 data for month: 2023-05-01 to 2023-06-01
Processing Sentinel-2 data for month: 2023-06-01 to 2023-07-01
Finished Sentinel-2 data extraction for months from index 24 to 29.
Total monthly Sentinel-2 DataFrames collected so far: 30


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (July 2023 to December 2023, indices 30-35)
chunk_start_idx = 30
chunk_end_idx = 36

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 30 to 35...
Processing Sentinel-2 data for month: 2023-07-01 to 2023-08-01
Processing Sentinel-2 data for month: 2023-08-01 to 2023-09-01
Processing Sentinel-2 data for month: 2023-09-01 to 2023-10-01


Processing Sentinel-2 data for month: 2023-10-01 to 2023-11-01
Processing Sentinel-2 data for month: 2023-11-01 to 2023-12-01
Processing Sentinel-2 data for month: 2023-12-01 to 2024-01-01
Finished Sentinel-2 data extraction for months from index 30 to 35.
Total monthly Sentinel-2 DataFrames collected so far: 36


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (January 2024 to June 2024, indices 36-41)
chunk_start_idx = 36
chunk_end_idx = 42

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 36 to 41...
Processing Sentinel-2 data for month: 2024-01-01 to 2024-02-01
Processing Sentinel-2 data for month: 2024-02-01 to 2024-03-01
Processing Sentinel-2 data for month: 2024-03-01 to 2024-04-01
Processing Sentinel-2 data for month: 2024-04-01 to 2024-05-01
Processing Sentinel-2 data for month: 2024-05-01 to 2024-06-01
Processing Sentinel-2 data for month: 2024-06-01 to 2024-07-01
Finished Sentinel-2 data extraction for months from index 36 to 41.
Total monthly Sentinel-2 DataFrames collected so far: 42


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (July 2024 to December 2024, indices 42-47)
chunk_start_idx = 42
chunk_end_idx = 48

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 42 to 47...
Processing Sentinel-2 data for month: 2024-07-01 to 2024-08-01
Processing Sentinel-2 data for month: 2024-08-01 to 2024-09-01
Processing Sentinel-2 data for month: 2024-09-01 to 2024-10-01
Processing Sentinel-2 data for month: 2024-10-01 to 2024-11-01


Processing Sentinel-2 data for month: 2024-11-01 to 2024-12-01
Processing Sentinel-2 data for month: 2024-12-01 to 2025-01-01
Finished Sentinel-2 data extraction for months from index 42 to 47.
Total monthly Sentinel-2 DataFrames collected so far: 48


In [ ]:
import ee
import pandas as pd

# 1. Ensure ee is authenticated and initialized
ee.Authenticate()
ee.Initialize(project='mangroove-startup')

# 2. Define the speckle filter function
def apply_speckle_filter(image):
    filtered_image = image.focal_median(radius=3)
    return filtered_image.copyProperties(image, image.propertyNames())

# 3. Define the fetch_features_in_chunks function
def fetch_features_in_chunks(feature_collection, image, chunk_size=250):
    total_features = feature_collection.size().getInfo()
    all_extracted_features = []

    for i in range(0, total_features, chunk_size):
        end_index = min(i + chunk_size, total_features)
        current_chunk_fc = feature_collection.toList(end_index).slice(i, end_index)
        current_chunk_fc = ee.FeatureCollection(current_chunk_fc)

        chunk_features = image.reduceRegions(
            reducer=ee.Reducer.first(),
            collection=current_chunk_fc,
            scale=10
        )

        try:
            chunk_list = chunk_features.getInfo()['features']
            all_extracted_features.extend(chunk_list)
        except ee.EEException as e:
            print(f"Error fetching chunk {i}-{end_index}: {e}")
            continue
    return all_extracted_features

# Re-create the Earth Engine FeatureCollection from the 'df' DataFrame (if not already present)
if 'mangrove_feature_collection' not in locals():
    mangrove_ee_features = []
    for index, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        mangrove_ee_features.append(ee.Feature(point))
    mangrove_feature_collection = ee.FeatureCollection(mangrove_ee_features)

# Initialize or ensure all_monthly_data is available
if 'all_monthly_data' not in globals():
    all_monthly_data = []

# 4. Set chunk indices for the next chunk of months
# The previous execution processed up to index 47. So, start from 48.
chunk_start_idx = 48
chunk_size = 6
# Ensure chunk_end_idx does not exceed the valid range for monthly_dates[i+1]
chunk_end_idx = min(chunk_start_idx + chunk_size, len(monthly_dates) - 1)

print(f"Continuing monthly data extraction for periods from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# 5. Loop through the months for the current chunk
for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    next_month_start = monthly_dates[i+1]

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing data for month: {start_date_str} to {end_date_str}")

    # 6b. Load Sentinel-1 GRD ImageCollection for backscatter
    sentinel1_backscatter = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) \
        .filter(ee.Filter.eq('productType', 'GRD'))

    # 6c. Apply speckle filter and create median composite
    if sentinel1_backscatter.size().getInfo() > 0:
        sentinel1_backscatter_filtered = sentinel1_backscatter.map(apply_speckle_filter)
        median_backscatter = sentinel1_backscatter_filtered.median().select(['VV', 'VH'])

        # 6d. Extract backscatter values
        backscatter_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_backscatter
        )

        backscatter_data = []
        for feature in backscatter_features:
            properties = feature['properties']
            backscatter_data.append({
                'VV_backscatter': properties.get('VV'),
                'VH_backscatter': properties.get('VH')
            })
        df_backscatter = pd.DataFrame(backscatter_data)
    else:
        df_backscatter = pd.DataFrame(columns=['VV_backscatter', 'VH_backscatter'], index=range(len(df)))
        df_backscatter = df_backscatter.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data


    # 6e. Load Sentinel-1 GRD_FLOAT ImageCollection for coherence
    sentinel1_coherence = ee.ImageCollection('COPERNICUS/S1_GRD_FLOAT') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))

    # 6f. Apply speckle filter and create median composite
    if sentinel1_coherence.size().getInfo() > 0:
        sentinel1_coherence_filtered = sentinel1_coherence.map(apply_speckle_filter)
        median_coherence = sentinel1_coherence_filtered.median().select(['VV', 'VH'])

        # 6g. Extract coherence values
        coherence_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            median_coherence
        )

        coherence_data = []
        for feature in coherence_features:
            properties = feature['properties']
            coherence_data.append({
                'VV_coherence': properties.get('VV'),
                'VH_coherence': properties.get('VH')
            })
        df_coherence = pd.DataFrame(coherence_data)
    else:
        df_coherence = pd.DataFrame(columns=['VV_coherence', 'VH_coherence'], index=range(len(df)))
        df_coherence = df_coherence.fillna(0.0) # Fill with 0 or NaN as appropriate for missing data

    # 6h. Combine data for the current month
    monthly_combined_df = pd.concat([df.reset_index(drop=True), df_backscatter, df_coherence], axis=1)
    monthly_combined_df['date'] = current_month_start
    # 6i. Append to all_monthly_data
    all_monthly_data.append(monthly_combined_df)

print(f"Finished processing months from index {chunk_start_idx} to {chunk_end_idx-1}.")
print(f"Total months processed so far: {len(all_monthly_data)}")

Continuing monthly data extraction for periods from index 48 to 53...
Processing data for month: 2025-01-01 to 2025-02-01
Processing data for month: 2025-02-01 to 2025-03-01
Processing data for month: 2025-03-01 to 2025-04-01
Processing data for month: 2025-04-01 to 2025-05-01
Processing data for month: 2025-05-01 to 2025-06-01
Processing data for month: 2025-06-01 to 2025-07-01
Finished processing months from index 48 to 53.
Total months processed so far: 6


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (July 2025 to December 2025, indices 54-59)
chunk_start_idx = 54
chunk_end_idx = 60

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 54 to 59...
Processing Sentinel-2 data for month: 2025-07-01 to 2025-08-01
Processing Sentinel-2 data for month: 2025-08-01 to 2025-09-01


Processing Sentinel-2 data for month: 2025-09-01 to 2025-10-01
Processing Sentinel-2 data for month: 2025-10-01 to 2025-11-01
Processing Sentinel-2 data for month: 2025-11-01 to 2025-12-01
Processing Sentinel-2 data for month: 2025-12-01 to 2026-01-01
Finished Sentinel-2 data extraction for months from index 54 to 59.
Total monthly Sentinel-2 DataFrames collected so far: 54


In [ ]:
import ee
import pandas as pd

# Ensure ee is authenticated and initialized. It was initialized in a previous step.
ee.Authenticate()
ee.Initialize(project='mangroove-startup') # Assuming this project is available and authenticated

# Define the chunk of months to process (January 2026, index 60)
chunk_start_idx = 60
chunk_end_idx = min(chunk_start_idx + 1, len(monthly_dates)) # Process only one month for the final chunk

print(f"Starting Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_dates[i]
    # Ensure next_month_start is within bounds of monthly_dates or calculated for the last month
    if i + 1 < len(monthly_dates):
        next_month_start = monthly_dates[i+1]
    else:
        # For the very last month, use the end of the month or a reasonable period
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing Sentinel-2 data for month: {start_date_str} to {end_date_str}")

    # Load Sentinel-2 ImageCollection and filter
    sentinel2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(start_date_str, end_date_str) \
        .filterBounds(mangrove_feature_collection) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

    # Check if the filtered ImageCollection contains any images
    if sentinel2_collection.size().getInfo() > 0:
        # Map the add_derived_bands function over the filtered ImageCollection
        processed_collection = sentinel2_collection.map(add_derived_bands)

        # Create a median composite image from the processed collection
        median_s2_image = processed_collection.median()

        # Select the specified bands from the median composite
        selected_s2_image = median_s2_image.select(s2_all_bands_to_extract)

        # Extract features using the utility function
        s2_features = fetch_features_in_chunks(
            mangrove_feature_collection,
            selected_s2_image
        )

        # Convert extracted features to a Pandas DataFrame
        s2_data_for_month = []
        for feature in s2_features:
            properties = feature['properties']
            # Ensure all expected bands are present, fill with 0.0 if not
            feature_dict = {band: properties.get(band, 0.0) for band in s2_all_bands_to_extract}
            s2_data_for_month.append(feature_dict)
        df_s2_data = pd.DataFrame(s2_data_for_month)
    else:
        # If no images, create an empty DataFrame with 0.0 for all bands
        print(f"No Sentinel-2 images found for {start_date_str} to {end_date_str} after filtering. Filling with 0.0.")
        df_s2_data = pd.DataFrame(columns=s2_all_bands_to_extract, index=range(len(df)))
        df_s2_data = df_s2_data.fillna(0.0) # Fill with 0.0 as placeholder for missing data

    # Combine with original df and add date column
    # The original 'df' contains latitude and longitude, which are the base coordinates.
    # Reset index of 'df' to ensure proper concatenation if it was modified.
    monthly_combined_s2_df = pd.concat([df.reset_index(drop=True)[['latitude', 'longitude']], df_s2_data], axis=1)
    monthly_combined_s2_df['date'] = current_month_start

    # Append to the global list
    all_monthly_s2_data.append(monthly_combined_s2_df)

print(f"Finished Sentinel-2 data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly Sentinel-2 DataFrames collected so far: {len(all_monthly_s2_data)}")

Starting Sentinel-2 data extraction for months from index 60 to 60...
Processing Sentinel-2 data for month: 2026-01-01 to 2026-02-01
Finished Sentinel-2 data extraction for months from index 60 to 60.
Total monthly Sentinel-2 DataFrames collected so far: 55


In [ ]:
import pandas as pd

# Concatenate all monthly DataFrames into a single DataFrame
final_s2_df = pd.concat(all_monthly_s2_data, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_s2_monthly = '/content/drive/MyDrive/STAGE 1/S2_UAE_Monthly.csv'
final_s2_df.to_csv(output_csv_path_s2_monthly, index=False)

print("Monthly Sentinel-2 data extraction complete.")
print(f"Final combined monthly Sentinel-2 data saved to: {output_csv_path_s2_monthly}")
print("Final DataFrame head:")
print(final_s2_df.head())
print(f"Final DataFrame shape: {final_s2_df.shape}")

Monthly Sentinel-2 data extraction complete.
Final combined monthly Sentinel-2 data saved to: /content/drive/MyDrive/STAGE 1/S2_UAE_Monthly.csv
Final DataFrame head:
    latitude  longitude     B1     B2     B3     B4     B5     B6     B7  \
0  25.739637  54.859193  221.5  259.0  223.5  121.0  132.5  106.5   96.0   
1  25.346367  53.574829  228.0  252.0  195.0  135.0  115.0  102.0   98.0   
2  25.329268  54.673023  354.0  394.0  311.0  220.5  211.5  174.0  174.0   
3  25.257074  54.744144  276.0  342.5  237.0  160.5  163.0  140.5  134.5   
4  25.241875  54.756695  291.5  332.5  252.5  154.0  159.5  133.0  120.0   

      B8  ...      NDWI      NDSI  Moisture_index   SWIR  \
0   91.5  ...  0.252204  0.313435        0.027688  108.5   
1   91.0  ...  0.401316  0.502645        0.074681   47.0   
2  143.5  ...  0.428417  0.573177        0.058043   85.0   
3  123.5  ...  0.360189  0.500299        0.061511   78.0   
4  102.5  ...  0.409061  0.507940        0.067917   69.0   

   Scene_classif

# GEDI UAE Monthly Data Extraction


In [ ]:
import pandas as pd

df=pd.read_csv('/content/drive/MyDrive/STAGE 1/S2_UAE_MANGROVE.CSV')
df=df[df['category'] == 'Mangrove'][['latitude', 'longitude']].reset_index(drop=True)
df.head()

,latitude,longitude
0,25.739637,54.859193
1,25.346367,53.574829
2,25.329268,54.673023
3,25.257074,54.744144
4,25.241875,54.756695


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions

# Ensure Earthdata is authenticated
# It was authenticated in the previous step, but re-authenticate for robustness if kernel restart occurs.
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# --- Re-add bbox definition for robustness ---
# Assuming df is available from previous cells. If not, it needs to be loaded here.
if 'df' not in globals():
    # This part should ideally not be reached if df is properly loaded in setup cells.
    # For absolute robustness, one might reload df here if it's the base of the coordinates.
    # Given the previous context, df is available from cell 'ovEpFF-XxXHr'
    print("WARNING: 'df' not found. Please ensure mangrove coordinates are loaded.")
    # df = pd.read_csv('/content/drive/MyDrive/STAGE 1/S2_UAE_MANGROVE.CSV')
    # df = df[df['category'] == 'Mangrove'][['latitude', 'longitude']].reset_index(drop=True)

min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# --- Re-add the definition of monthly_gedi_dates for robustness ---
# Define start and end dates for monthly GEDI data extraction
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
# Generate monthly date ranges
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")
# --- End of re-added monthly_gedi_dates definition ---

# --- Re-add temporary directory definitions and creation for robustness ---
out_dir_l2a_tmp = "/content/gedi_l2a_tmp"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist.")
# --- End of re-added temporary directory definitions ---

# --- Re-define the GEDI extraction functions for robustness ---
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for 'rh' and 'geolocation' groups
                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    # Check for required datasets within their respective groups
                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        # Ensure all arrays have the same length before filtering
                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue # Skip this beam if no data after flattening/length check

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        # Filter out fill values (-9999 or NaN) using vectorized operations
                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function defined.")

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    # Update required_direct_keys based on inspection results
                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        # Ensure all arrays have the same length before filtering
                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue # Skip this beam if no data after flattening/length check

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        # Filter out fill values (-9999 or NaN) for all relevant fields
                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function defined.")

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                # Check if 'agbd' exists before trying to access it
                if "agbd" not in g:
                    continue

                # Safely access datasets, assuming they exist under the beam group directly
                # Based on previous code in the notebook, these were accessed directly under 'g'
                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    # Filter out fill values (-9999 or NaN) for AGBD
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function defined.")
# --- End of re-defined GEDI extraction functions ---

# Define the chunk of months to process (January 2021 to June 2021, indices 0-5)
chunk_start_idx = 0
chunk_end_idx = 6

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# Initialize a list to store all monthly GEDI dataframes
# This list should already be initialized from the setup cell. Re-initializing for robustness.
if 'all_monthly_gedi_shots' not in globals():
    all_monthly_gedi_shots = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        # Clean up downloaded files
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        # Clean up downloaded files
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        # Clean up downloaded files
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots
final_gedi_df = pd.concat(all_monthly_gedi_shots, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_1 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_1.csv'
final_gedi_df.to_csv(output_csv_path_gedi_chunk_1, index=False)

print("GEDI data extraction chunk 1 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_1}")
print("Final DataFrame head:")
print(final_gedi_df.head())
print(f"Final DataFrame shape: {final_gedi_df.shape}")

# Clean up temporary download directories at the end of the entire process
import shutil
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist.
GEDI L2A extraction function defined.
GEDI L2B extraction function defined.
GEDI L4A extraction function defined.
Starting GEDI data extraction for months from index 0 to 5...
Processing GEDI data for month: 2021-01-01 to 2021-02-01
Searching for GEDI L2A granules for 2021-01-01...
Found 14 GEDI L2A granules.
Searching for GEDI L2B granules for 2021-01-01...
Found 14 GEDI L2B granules.
Searching for GEDI L4A granules for 2021-01-01...
Found 14 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/14 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/14 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/14 [00:00<?, ?it/s]

Combined GEDI data for 2021-06-01 has 14611409 shots.
Performed nearest-neighbor join for 2021-06-01.
Finished GEDI data extraction for months from index 0 to 5.
GEDI data extraction chunk 1 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_1.csv
Final DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  GEDI_biomass_Mg_ha  \
0  25.739637  54.859193       NaN       NaN          NaN            0.742466   
1  25.346367  53.574829   0.00752  1.394708     0.003753                 NaN   
2  25.329268  54.673023       NaN       NaN          NaN            0.834731   
3  25.257074  54.744144       NaN       NaN          NaN            0.834731   
4  25.241875  54.756695       NaN       NaN          NaN            0.834731   

        date  
0 2021-01-01  
1 2021-01-01  
2 2021-01-01  
3 2021-01-01  
4 2021-01-01  
Final DataFrame shape: (24060, 7)
Temporary download directories cleaned up.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk2"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk2"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk2"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 2.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices
chunk_start_idx = 6
chunk_end_idx = 12

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (July 2021 - December 2021)...")

# 7. Initialize an empty list to store all monthly GEDI dataframes
all_monthly_gedi_shots_chunk2 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk2.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk2
final_gedi_df_chunk2 = pd.concat(all_monthly_gedi_shots_chunk2, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_2 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_2.csv'
final_gedi_df_chunk2.to_csv(output_csv_path_gedi_chunk_2, index=False)

print("GEDI data extraction chunk 2 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_2}")
print("Final DataFrame head:")
print(final_gedi_df_chunk2.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk2.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 2.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 2.
Starting GEDI data extraction for months from index 6 to 11 (July 2021 - December 2021)...
Processing GEDI data for month: 2021-07-01 to 2021-08-01
Searching for GEDI L2A granules for 2021-07-01...
Found 17 GEDI L2A granules.
Searching for GEDI L2B granules for 2021-07-01...
Found 17 GEDI L2B granules.
Searching for GEDI L4A granules for 2021-07-01...
Found 17 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/23 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/23 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/23 [00:00<?, ?it/s]

Combined GEDI data for 2021-12-01 has 18523294 shots.
Performed nearest-neighbor join for 2021-12-01.
Finished GEDI data extraction for months from index 6 to 11.
GEDI data extraction chunk 2 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_2.csv
Final DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  GEDI_biomass_Mg_ha  \
0  25.739637  54.859193  0.004944  1.099477     0.002469                 NaN   
1  25.346367  53.574829  0.051070  1.325721     0.025212                 NaN   
2  25.329268  54.673023  0.060725  1.184226     0.029906                 NaN   
3  25.257074  54.744144  0.032674  1.035525     0.016204                 NaN   
4  25.241875  54.756695  0.031433  1.235133     0.015593                 NaN   

        date  
0 2021-07-01  
1 2021-07-01  
2 2021-07-01  
3 2021-07-01  
4 2021-07-01  
Final DataFrame shape: (24060, 7)
Temporary download directories cleaned up for chunk 2.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 3
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk3"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk3"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk3"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 3.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 3: January 2022 to June 2022 (indices 12-17)
chunk_start_idx = 12
chunk_end_idx = 18

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (January 2022 - June 2022)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk3 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk3.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk3
final_gedi_df_chunk3 = pd.concat(all_monthly_gedi_shots_chunk3, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_3 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_3.csv'
final_gedi_df_chunk3.to_csv(output_csv_path_gedi_chunk_3, index=False)

print("GEDI data extraction chunk 3 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_3}")
print("Final DataFrame head:")
print(final_gedi_df_chunk3.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk3.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 3.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 3.
Starting GEDI data extraction for months from index 12 to 17 (January 2022 - June 2022)...
Processing GEDI data for month: 2022-01-01 to 2022-02-01
Searching for GEDI L2A granules for 2022-01-01...
Found 18 GEDI L2A granules.
Searching for GEDI L2B granules for 2022-01-01...
Found 18 GEDI L2B granules.
Searching for GEDI L4A granules for 2022-01-01...
Found 18 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

Combined GEDI data for 2022-06-01 has 15004395 shots.
Performed nearest-neighbor join for 2022-06-01.
Finished GEDI data extraction for months from index 12 to 17.
GEDI data extraction chunk 3 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_3.csv
Final DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  GEDI_biomass_Mg_ha  \
0  25.739637  54.859193  0.001543  0.636514     0.000771                 NaN   
1  25.346367  53.574829  0.557622  1.541278     0.243268                 NaN   
2  25.329268  54.673023       NaN       NaN          NaN            0.247300   
3  25.257074  54.744144       NaN       NaN          NaN            0.273898   
4  25.241875  54.756695  0.124970  1.181846     0.060570                 NaN   

        date  
0 2022-01-01  
1 2022-01-01  
2 2022-01-01  
3 2022-01-01  
4 2022-01-01  
Final DataFrame shape: (24060, 7)
Temporary download directories cleaned up for chunk 3.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 4
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk4"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk4"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk4"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 4.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 4: July 2022 to December 2022 (indices 18-23)
chunk_start_idx = 18
chunk_end_idx = 24

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (July 2022 - December 2022)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk4 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk4.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk4
final_gedi_df_chunk4 = pd.concat(all_monthly_gedi_shots_chunk4, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_4 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_4.csv'
final_gedi_df_chunk4.to_csv(output_csv_path_gedi_chunk_4, index=False)

print("GEDI data extraction chunk 4 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_4}")
print("Final DataFrame head:")
print(final_gedi_df_chunk4.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk4.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 4.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 4.
Starting GEDI data extraction for months from index 18 to 23 (July 2022 - December 2022)...
Processing GEDI data for month: 2022-07-01 to 2022-08-01
Searching for GEDI L2A granules for 2022-07-01...
Found 22 GEDI L2A granules.
Searching for GEDI L2B granules for 2022-07-01...
Found 22 GEDI L2B granules.
Searching for GEDI L4A granules for 2022-07-01...
Found 22 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/19 [00:00<?, ?it/s]

Combined GEDI data for 2022-12-01 has 17605219 shots.
Performed nearest-neighbor join for 2022-12-01.
Finished GEDI data extraction for months from index 18 to 23.
GEDI data extraction chunk 4 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_4.csv
Final DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  GEDI_biomass_Mg_ha  \
0  25.739637  54.859193       NaN       NaN          NaN            0.377637   
1  25.346367  53.574829       NaN       NaN          NaN            0.544866   
2  25.329268  54.673023       NaN       NaN          NaN            0.787927   
3  25.257074  54.744144       NaN       NaN          NaN            0.787927   
4  25.241875  54.756695       NaN       NaN          NaN            0.787927   

        date  
0 2022-07-01  
1 2022-07-01  
2 2022-07-01  
3 2022-07-01  
4 2022-07-01  
Final DataFrame shape: (24060, 7)
Temporary download directories cleaned up for chunk 4.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 5
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk5"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk5"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk5"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 5.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 5: January 2023 to June 2023 (indices 24-29)
chunk_start_idx = 24
chunk_end_idx = 30

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (January 2023 - June 2023)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk5 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk5.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk5
final_gedi_df_chunk5 = pd.concat(all_monthly_gedi_shots_chunk5, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_5 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_5.csv'
final_gedi_df_chunk5.to_csv(output_csv_path_gedi_chunk_5, index=False)

print("GEDI data extraction chunk 5 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_5}")
print("Final DataFrame head:")
print(final_gedi_df_chunk5.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk5.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 5.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 5.
Starting GEDI data extraction for months from index 24 to 29 (January 2023 - June 2023)...
Processing GEDI data for month: 2023-01-01 to 2023-02-01
Searching for GEDI L2A granules for 2023-01-01...
Found 24 GEDI L2A granules.
Searching for GEDI L2B granules for 2023-01-01...
Found 24 GEDI L2B granules.
Searching for GEDI L4A granules for 2023-01-01...
Found 24 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/10 [00:00<?, ?it/s]

Combined GEDI data for 2023-03-01 has 10968735 shots.
Performed nearest-neighbor join for 2023-03-01.
Processing GEDI data for month: 2023-04-01 to 2023-05-01
Searching for GEDI L2A granules for 2023-04-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2023-04-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2023-04-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2023-04-01.
Skipping nearest-neighbor join for 2023-04-01 as no GEDI data was collected.
Processing GEDI data for month: 2023-05-01 to 2023-06-01
Searching for GEDI L2A granules for 2023-05-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2023-05-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2023-05-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2023-05-01.
Skipping nearest-neighbor join for 2023-05-01 as no GEDI data was collected.
Processing GEDI data for month: 2023-06-01 to 2023-07-01
Searching for GEDI L2A granules f

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 6
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk6"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk6"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk6"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 6.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 6: July 2023 to December 2023 (indices 30-35)
chunk_start_idx = 30
chunk_end_idx = 36

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (July 2023 - December 2023)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk6 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk6.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk6
final_gedi_df_chunk6 = pd.concat(all_monthly_gedi_shots_chunk6, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_6 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_6.csv'
final_gedi_df_chunk6.to_csv(output_csv_path_gedi_chunk_6, index=False)

print("GEDI data extraction chunk 6 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_6}")
print("Final DataFrame head:")
print(final_gedi_df_chunk6.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk6.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 6.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 6.
Starting GEDI data extraction for months from index 30 to 35 (July 2023 - December 2023)...
Processing GEDI data for month: 2023-07-01 to 2023-08-01
Searching for GEDI L2A granules for 2023-07-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2023-07-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2023-07-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2023-07-01.
Skipping nearest-neighbor join for 2023-07-01 as no GEDI data was collected.
Processing GEDI data for month: 2023-08-01 to 2023-09-01
Searching for GEDI L2A granules for 2023-08-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2023-08-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A gra

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 7
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk7"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk7"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk7"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 7.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 7: January 2024 to June 2024 (indices 36-41)
chunk_start_idx = 36
chunk_end_idx = 42

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (January 2024 - June 2024)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk7 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk7.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk7
final_gedi_df_chunk7 = pd.concat(all_monthly_gedi_shots_chunk7, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_7 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_7.csv'
final_gedi_df_chunk7.to_csv(output_csv_path_gedi_chunk_7, index=False)

print("GEDI data extraction chunk 7 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_7}")
print("Final DataFrame head:")
print(final_gedi_df_chunk7.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk7.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 7.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 7.
Starting GEDI data extraction for months from index 36 to 41 (January 2024 - June 2024)...
Processing GEDI data for month: 2024-01-01 to 2024-02-01
Searching for GEDI L2A granules for 2024-01-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2024-01-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2024-01-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2024-01-01.
Skipping nearest-neighbor join for 2024-01-01 as no GEDI data was collected.
Processing GEDI data for month: 2024-02-01 to 2024-03-01
Searching for GEDI L2A granules for 2024-02-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2024-02-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A gran

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/20 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/20 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/20 [00:00<?, ?it/s]

Combined GEDI data for 2024-06-01 has 18104303 shots.
Performed nearest-neighbor join for 2024-06-01.
Finished GEDI data extraction for months from index 36 to 41.
GEDI data extraction chunk 7 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_7.csv
Final DataFrame head:
    latitude  longitude       date  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  \
0  25.739637  54.859193 2024-01-01       NaN       NaN          NaN   
1  25.346367  53.574829 2024-01-01       NaN       NaN          NaN   
2  25.329268  54.673023 2024-01-01       NaN       NaN          NaN   
3  25.257074  54.744144 2024-01-01       NaN       NaN          NaN   
4  25.241875  54.756695 2024-01-01       NaN       NaN          NaN   

   GEDI_biomass_Mg_ha  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4                 NaN  
Final DataFrame shape: (24060, 7)
Temporary download directories cleaned up for chunk 7.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 8
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk8"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk8"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk8"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 8.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 8: July 2024 to December 2024 (indices 42-47)
chunk_start_idx = 42
chunk_end_idx = 48

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (July 2024 - December 2024)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk8 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk8.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk8
final_gedi_df_chunk8 = pd.concat(all_monthly_gedi_shots_chunk8, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_8 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_8.csv'
final_gedi_df_chunk8.to_csv(output_csv_path_gedi_chunk_8, index=False)

print("GEDI data extraction chunk 8 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_8}")
print("Final DataFrame head:")
print(final_gedi_df_chunk8.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk8.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 8.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 8.
Starting GEDI data extraction for months from index 42 to 47 (July 2024 - December 2024)...
Processing GEDI data for month: 2024-07-01 to 2024-08-01
Searching for GEDI L2A granules for 2024-07-01...
Found 22 GEDI L2A granules.
Searching for GEDI L2B granules for 2024-07-01...
Found 22 GEDI L2B granules.
Searching for GEDI L4A granules for 2024-07-01...
Found 22 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/22 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/22 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/22 [00:00<?, ?it/s]

Combined GEDI data for 2024-12-01 has 19765120 shots.
Performed nearest-neighbor join for 2024-12-01.
Finished GEDI data extraction for months from index 42 to 47.
GEDI data extraction chunk 8 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_8.csv
Final DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  GEDI_biomass_Mg_ha  \
0  25.739637  54.859193       NaN       NaN          NaN            0.776436   
1  25.346367  53.574829       NaN       NaN          NaN            0.776436   
2  25.329268  54.673023       NaN       NaN          NaN            0.957603   
3  25.257074  54.744144       NaN       NaN          NaN            0.776436   
4  25.241875  54.756695       NaN       NaN          NaN            0.776436   

        date  
0 2024-07-01  
1 2024-07-01  
2 2024-07-01  
3 2024-07-01  
4 2024-07-01  
Final DataFrame shape: (24060, 7)
Temporary download directories cleaned up for chunk 8.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 9
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk9"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk9"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk9"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 9.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 9: January 2025 to June 2025 (indices 48-53)
chunk_start_idx = 48
chunk_end_idx = 54

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (January 2025 - June 2025)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk9 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk9.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk9
final_gedi_df_chunk9 = pd.concat(all_monthly_gedi_shots_chunk9, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_9 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_9.csv'
final_gedi_df_chunk9.to_csv(output_csv_path_gedi_chunk_9, index=False)

print("GEDI data extraction chunk 9 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_9}")
print("Final DataFrame head:")
print(final_gedi_df_chunk9.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk9.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 9.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 9.
Starting GEDI data extraction for months from index 48 to 53 (January 2025 - June 2025)...
Processing GEDI data for month: 2025-01-01 to 2025-02-01
Searching for GEDI L2A granules for 2025-01-01...
Found 21 GEDI L2A granules.
Searching for GEDI L2B granules for 2025-01-01...
Found 21 GEDI L2B granules.
Searching for GEDI L4A granules for 2025-01-01...
Found 21 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

...

Combined GEDI data for 2025-02-01 has 14788927 shots.
Performed nearest-neighbor join for 2025-02-01.
Processing GEDI data for month: 2025-03-01 to 2025-04-01
Searching for GEDI L2A granules for 2025-03-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2025-03-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2025-03-01...
Found 11 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/11 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/11 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/11 [00:00<?, ?it/s]

Combined GEDI data for 2025-03-01 has 7131560 shots.
Performed nearest-neighbor join for 2025-03-01.
Processing GEDI data for month: 2025-04-01 to 2025-05-01
Searching for GEDI L2A granules for 2025-04-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2025-04-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2025-04-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2025-04-01.
Skipping nearest-neighbor join for 2025-04-01 as no GEDI data was collected.
Processing GEDI data for month: 2025-05-01 to 2025-06-01
Searching for GEDI L2A granules for 2025-05-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2025-05-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2025-05-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2025-05-01.
Skipping nearest-neighbor join for 2025-05-01 as no GEDI data was collected.
Processing GEDI data for month: 2025-06-01 to 2025-07-01
Searching for GEDI L2A granules fo

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 10
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk10"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk10"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk10"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 10.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 10: July 2025 to December 2025 (indices 54-59)
chunk_start_idx = 54
chunk_end_idx = 60

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (July 2025 - December 2025)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk10 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk10.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk10
final_gedi_df_chunk10 = pd.concat(all_monthly_gedi_shots_chunk10, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_10 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_10.csv'
final_gedi_df_chunk10.to_csv(output_csv_path_gedi_chunk_10, index=False)

print("GEDI data extraction chunk 10 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_10}")
print("Final DataFrame head:")
print(final_gedi_df_chunk10.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk10.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 10.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 10.
Starting GEDI data extraction for months from index 54 to 59 (July 2025 - December 2025)...
Processing GEDI data for month: 2025-07-01 to 2025-08-01
Searching for GEDI L2A granules for 2025-07-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2025-07-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2025-07-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2025-07-01.
Skipping nearest-neighbor join for 2025-07-01 as no GEDI data was collected.
Processing GEDI data for month: 2025-08-01 to 2025-09-01
Searching for GEDI L2A granules for 2025-08-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2025-08-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A gr

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py # Ensure h5py is imported for extraction functions
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Ensure df is available from previous cells. If not, it needs to be loaded here.
# Based on kernel state, df is available.
min_lat = df.latitude.min()
max_lat = df.latitude.max()
min_lon = df.longitude.min()
max_lon = df.longitude.max()
bbox = (min_lon, min_lat, max_lon, max_lat)
print(f"Bounding box derived: {bbox}")

# 3. Regenerate the `monthly_gedi_dates` list
start_date_gedi = '2021-01-01'
end_date_gedi = '2026-01-31'
monthly_gedi_dates = pd.date_range(start=start_date_gedi, end=end_date_gedi, freq='MS').tolist()
print(f"Generated {len(monthly_gedi_dates)} monthly GEDI date ranges.")

# 4. Redefine temporary directory paths and ensure they exist for chunk 11 (final chunk)
out_dir_l2a_tmp = "/content/gedi_l2a_tmp_chunk11"
out_dir_l2b_tmp = "/content/gedi_l2b_tmp_chunk11"
out_dir_l4a_tmp = "/content/gedi_l4a_tmp_chunk11"

os.makedirs(out_dir_l2a_tmp, exist_ok=True)
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print("Temporary GEDI data directories ensured to exist for chunk 11.")

# 5. Redefine the GEDI extraction functions for robustness
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'rh' in g and isinstance(g['rh'], h5py.Group) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group):

                    rh_group = g['rh']
                    geo_group = g['geolocation']

                    required_rh_keys = ['rh100']
                    required_geo_keys = ['elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode']

                    if all(key in rh_group for key in required_rh_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        canopy_height = np.asarray(rh_group["rh100"][:], dtype=np.float64).flatten()
                        elevation = np.asarray(geo_group["elev_lowestmode"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(canopy_height), len(elevation), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        canopy_height = canopy_height[:min_len]
                        elevation = elevation[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (canopy_height > -9999) & (~np.isnan(canopy_height)) & \
                                        (elevation > -9999) & (~np.isnan(elevation)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "canopy_height_rh100": float(ch_val),
                                    "elevation_lowestmode": float(elev_val)
                                }
                                for ch_val, elev_val, lat_val, lon_val in zip(
                                    canopy_height[valid_indices],
                                    elevation[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                if 'geolocation' in g and isinstance(g['geolocation'], h5py.Group):
                    geo_group = g['geolocation']

                    required_direct_keys = ['pai', 'fhd_normal', 'cover']
                    required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                    if all(key in g for key in required_direct_keys) and \
                       all(key in geo_group for key in required_geo_keys):

                        pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                        fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                        cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                        lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                        lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                        min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                        if min_len == 0:
                            continue

                        pai = pai[:min_len]
                        fhd = fhd[:min_len]
                        cover = cover[:min_len]
                        lat = lat[:min_len]
                        lon = lon[:min_len]

                        valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                        (fhd > -9999) & (~np.isnan(fhd)) & \
                                        (cover > -9999) & (~np.isnan(cover)) & \
                                        (lat > -9999) & (~np.isnan(lat)) & \
                                        (lon > -9999) & (~np.isnan(lon))

                        if valid_indices.any():
                            rows.extend([
                                {
                                    "latitude": float(lat_val),
                                    "longitude": float(lon_val),
                                    "PAI_GEDI": float(pai_val),
                                    "FHD_GEDI": float(fhd_val),
                                    "FCOVER_GEDI": float(cover_val)
                                }
                                for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                    pai[valid_indices],
                                    fhd[valid_indices],
                                    cover[valid_indices],
                                    lat[valid_indices],
                                    lon[valid_indices]
                                )
                            ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]
                if "agbd" not in g:
                    continue

                agbd = g["agbd"][:]
                lat = g["lat_lowestmode"][:]
                lon = g["lon_lowestmode"][:]

                for i in range(len(agbd)):
                    if agbd[i] > -9999 and not np.isnan(agbd[i]):
                        rows.append({
                            "latitude": float(lat[i]),
                            "longitude": float(lon[i]),
                            "biomass_Mg_ha": float(agbd[i])
                        })
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)

# 6. Set chunk indices for Chunk 11: January 2026 (indices 60-60)
chunk_start_idx = 60
chunk_end_idx = min(chunk_start_idx + 1, len(monthly_gedi_dates)) # Process only one month for the final chunk

print(f"Starting GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1} (January 2026)...")

# 7. Initialize a list to store all monthly GEDI dataframes for this chunk
all_monthly_gedi_shots_chunk11 = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l2a_data = []
    monthly_l2b_data = []
    monthly_l4a_data = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a = extract_l2a(fpath)
            if not df_l2a.empty:
                monthly_l2a_data.append(df_l2a)
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b:
            df_l2b = extract_l2b(fpath)
            if not df_l2b.empty:
                monthly_l2b_data.append(df_l2b)
        for f in os.listdir(out_dir_l2b_tmp):
            os.remove(os.path.join(out_dir_l2b_tmp, f))

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        files_l4a = earthaccess.download(l4a_granules[:100], out_dir_l4a_tmp)
        for fpath in files_l4a:
            df_l4a = extract_l4a(fpath)
            if not df_l4a.empty:
                monthly_l4a_data.append(df_l4a)
        for f in os.listdir(out_dir_l4a_tmp):
            os.remove(os.path.join(out_dir_l4a_tmp, f))

    # Concatenate all available GEDI data for the current month
    current_month_gedi_data_frames = []
    if monthly_l2a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2a_data, ignore_index=True))
    if monthly_l2b_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l2b_data, ignore_index=True))
    if monthly_l4a_data:
        current_month_gedi_data_frames.append(pd.concat(monthly_l4a_data, ignore_index=True))

    combined_gedi_monthly_df = pd.DataFrame()
    if current_month_gedi_data_frames:
        combined_gedi_monthly_df = pd.concat(current_month_gedi_data_frames, ignore_index=True)
        print(f"Combined GEDI data for {start_date_str} has {len(combined_gedi_monthly_df)} shots.")
    else:
        print(f"No GEDI data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi = df.copy()

    # Perform nearest-neighbor join if GEDI data is available
    if not combined_gedi_monthly_df.empty:
        X_gedi_monthly = combined_gedi_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi
        if "canopy_height_rh100" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_canopy_height_rh100"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        if "elevation_lowestmode" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_elevation_lowestmode"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values

        # Add L2B data to df_monthly_gedi
        if "PAI_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_PAI"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        if "FHD_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FHD"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        if "FCOVER_GEDI" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_FCOVER"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values

        # Add L4A data to df_monthly_gedi
        if "biomass_Mg_ha" in combined_gedi_monthly_df.columns:
            df_monthly_gedi["GEDI_biomass_Mg_ha"] = combined_gedi_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi['date'] = current_month_start
    all_monthly_gedi_shots_chunk11.append(df_monthly_gedi)

print(f"Finished GEDI data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")

# After the loop, concatenate all DataFrames in all_monthly_gedi_shots_chunk11
final_gedi_df_chunk11 = pd.concat(all_monthly_gedi_shots_chunk11, ignore_index=True)

# Save the final DataFrame to a CSV file
output_csv_path_gedi_chunk_11 = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_11.csv'
final_gedi_df_chunk11.to_csv(output_csv_path_gedi_chunk_11, index=False)

print("GEDI data extraction chunk 11 complete.")
print(f"Final combined GEDI data saved to: {output_csv_path_gedi_chunk_11}")
print("Final DataFrame head:")
print(final_gedi_df_chunk11.head())
print(f"Final DataFrame shape: {final_gedi_df_chunk11.shape}")

# Clean up temporary download directories at the end of the entire process
if os.path.exists(out_dir_l2a_tmp): shutil.rmtree(out_dir_l2a_tmp)
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print("Temporary download directories cleaned up for chunk 11.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
Bounding box derived: (51.8658315144, 22.79106265712945, 56.6539543896, 26.195602094277675)
Generated 61 monthly GEDI date ranges.
Temporary GEDI data directories ensured to exist for chunk 11.
Starting GEDI data extraction for months from index 60 to 60 (January 2026)...
Processing GEDI data for month: 2026-01-01 to 2026-02-01
Searching for GEDI L2A granules for 2026-01-01...
Found 0 GEDI L2A granules.
Searching for GEDI L2B granules for 2026-01-01...
Found 0 GEDI L2B granules.
Searching for GEDI L4A granules for 2026-01-01...
Found 0 GEDI L4A granules.
No GEDI data collected for 2026-01-01.
Skipping nearest-neighbor join for 2026-01-01 as no GEDI data was collected.
Finished GEDI data extraction for months from index 60 to 60.
GEDI data extraction chunk 11 complete.
Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_11.csv
Final DataFrame head:
    latitude  longitude      

In [ ]:
import pandas as pd
import os

# Define the paths to all chunked GEDI CSV files
gedi_chunk_files = [
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_1.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_2.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_3.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_4.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_5.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_6.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_7.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_8.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_9.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_10.csv',
    '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_11.csv'
]

# Initialize an empty list to hold dataframes from each chunk
all_gedi_dfs = []

# Loop through the files, read them, and append to the list
print("Concatenating all GEDI monthly chunks...")
for file_path in gedi_chunk_files:
    if os.path.exists(file_path):
        try:
            df_chunk = pd.read_csv(file_path)
            all_gedi_dfs.append(df_chunk)
            print(f"Loaded {file_path} with shape {df_chunk.shape}")
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    else:
        print(f"Warning: {file_path} not found. Skipping.")

# Concatenate all dataframes into a single final dataframe
if all_gedi_dfs:
    final_gedi_df = pd.concat(all_gedi_dfs, ignore_index=True)

    # Sort by date and then by latitude/longitude to maintain consistency
    final_gedi_df = final_gedi_df.sort_values(by=['date', 'latitude', 'longitude']).reset_index(drop=True)

    # Define the final output path
    output_csv_path_gedi_final = '/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly.csv'

    # Save the final DataFrame to a CSV file
    final_gedi_df.to_csv(output_csv_path_gedi_final, index=False)

    print("\nFinal GEDI data compilation complete.")
    print(f"Final combined GEDI data saved to: {output_csv_path_gedi_final}")
    print("Final DataFrame head:")
    print(final_gedi_df.head())
    print(f"Final DataFrame shape: {final_gedi_df.shape}")
else:
    print("No GEDI data chunks were loaded. Final DataFrame could not be created.")


Concatenating all GEDI monthly chunks...
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_1.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_2.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_3.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_4.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_5.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_6.csv with shape (24060, 3)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_7.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_8.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_9.csv with shape (24060, 7)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_10.csv with shape (24060, 3)
Loaded /content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly_chunk_11.csv 

# S1 S2 GEDI Combining

In [ ]:
import pandas as pd

s1_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/S1_UAE_Monthly.csv')
s1_df['date'] = pd.to_datetime(s1_df['date'])

s2_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/S2_UAE_Monthly.csv')
s2_df['date'] = pd.to_datetime(s2_df['date'])

gedi_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_UAE_Monthly.csv')
gedi_df['date'] = pd.to_datetime(gedi_df['date'])

s1_df = s1_df.sort_values(by=['latitude', 'longitude', 'date']).reset_index(drop=True)
s2_df = s2_df.sort_values(by=['latitude', 'longitude', 'date']).reset_index(drop=True)
gedi_df = gedi_df.sort_values(by=['latitude', 'longitude', 'date']).reset_index(drop=True)


In [ ]:
# Drop common columns from s2_df and gedi_df to avoid duplication during concatenation
s2_df_unique = s2_df.drop(columns=['latitude', 'longitude', 'date'])
gedi_df_unique = gedi_df.drop(columns=['latitude', 'longitude', 'date'])

print(s2_df_unique.shape)
print(gedi_df_unique.shape)

(244610, 23)
(244610, 4)


In [ ]:
final_merged_side_by_side_df = pd.concat([s1_df, s2_df_unique, gedi_df_unique], axis=1)
output_csv_path_merged_sbs = '/content/drive/MyDrive/STAGE 1/S1_S2_GEDI_UAE_Monthly.csv'
final_merged_side_by_side_df.to_csv(output_csv_path_merged_sbs, index=False)

print(f"Side-by-side merged data saved to: {output_csv_path_merged_sbs}")
print("Final Merged (Side-by-Side) DataFrame head:")
print(final_merged_side_by_side_df.head())
print(f"Final Merged (Side-by-Side) DataFrame shape: {final_merged_side_by_side_df.shape}")

Side-by-side merged data saved to: /content/drive/MyDrive/STAGE 1/S1_S2_GEDI_UAE_Monthly.csv
Final Merged (Side-by-Side) DataFrame head:
    latitude  longitude  VV_backscatter  VH_backscatter  VV_coherence  \
0  22.791063  54.246296      -24.314251      -31.478721      0.003703   
1  22.791063  54.246296      -25.507535      -30.963398      0.002819   
2  22.791063  54.246296      -25.626493      -31.186509      0.002737   
3  22.791063  54.246296      -23.189345      -31.998171      0.004816   
4  22.791063  54.246296      -22.832819      -25.969246      0.005209   

   VH_coherence       date     B1      B2      B3  ...    SWIR  \
0      0.000711 2021-01-01  755.0  1004.5  1856.5  ...  6618.0   
1      0.000801 2021-02-01  743.0  1055.0  1854.0  ...  6711.0   
2      0.000761 2021-03-01  693.5  1009.0  1882.0  ...  6833.5   
3      0.000642 2021-04-01  733.0  1026.0  1736.0  ...  6830.0   
4      0.002530 2021-05-01  756.5  1032.0  1804.0  ...  6971.5   

   Scene_classification_map

# GEDI L2A UAE Monthly

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh100', 'rh98', 'rh92',
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    canopy_height_rh100 = np.asarray(g["rh100"][:], dtype=np.float64).flatten()
                    canopy_height_rh98 = np.asarray(g["rh98"][:], dtype=np.float64).flatten()
                    canopy_height_rh92 = np.asarray(g["rh92"][:], dtype=np.float64).flatten()
                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (January 2021 to June 2021, indices 0-5)
chunk_start_idx = 0
chunk_end_idx = 6

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a:
            df_l2a_extracted = extract_l2a(fpath)
            if not df_l2a_extracted.empty:
                monthly_l2a_shots.append(df_l2a_extracted)
        # Clean up downloaded files after processing
        for f in os.listdir(out_dir_l2a_tmp):
            os.remove(os.path.join(out_dir_l2a_tmp, f))

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

# No final save yet, as this is just the first chunk of L2A data

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 0 to 5...
Processing GEDI L2A data for month: 2021-01-01 to 2021-02-01
Searching for GEDI L2A granules for 2021-01-01...
Found 12 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

No GEDI L2A data collected for 2021-01-01.
Skipping nearest-neighbor join for 2021-01-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2021-02-01 to 2021-03-01
Searching for GEDI L2A granules for 2021-02-01...
Found 11 GEDI L2A granules.


...

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

No GEDI L2A data collected for 2021-05-01.
Skipping nearest-neighbor join for 2021-05-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2021-06-01 to 2021-07-01
Searching for GEDI L2A granules for 2021-06-01...
Found 14 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (July 2021 to December 2021, indices 6-11)
chunk_start_idx = 6
chunk_end_idx = 12

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 6 to 11...
Processing GEDI L2A data for month: 2021-07-01 to 2021-08-01
Searching for GEDI L2A granules for 2021-07-01...
Found 16 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2021-07-01 has 6423432 shots.
Performed nearest-neighbor join for 2021-07-01.
Processing GEDI L2A data for month: 2021-08-01 to 2021-09-01
Searching for GEDI L2A granules for 2021-08-01...
Found 22 GEDI L2A granules.


...

Combined GEDI L2A data for 2021-11-01 has 6683262 shots.
Performed nearest-neighbor join for 2021-11-01.
Processing GEDI L2A data for month: 2021-12-01 to 2022-01-01
Searching for GEDI L2A granules for 2021-12-01...
Found 21 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2021-12-01 has 6316898 shots.
Performed nearest-neighbor join for 2021-12-01.
Finished GEDI L2A data extraction for months from index 6 to 11.
Total monthly GEDI L2A DataFrames collected so far: 24


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (January 2022 to June 2022, indices 12-17)
chunk_start_idx = 12
chunk_end_idx = 18

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 12 to 17...
Processing GEDI L2A data for month: 2022-01-01 to 2022-02-01
Searching for GEDI L2A granules for 2022-01-01...
Found 17 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2022-01-01 has 6685378 shots.
Performed nearest-neighbor join for 2022-01-01.
Processing GEDI L2A data for month: 2022-02-01 to 2022-03-01
Searching for GEDI L2A granules for 2022-02-01...
Found 15 GEDI L2A granules.


...

Combined GEDI L2A data for 2022-05-01 has 6352624 shots.
Performed nearest-neighbor join for 2022-05-01.
Processing GEDI L2A data for month: 2022-06-01 to 2022-07-01
Searching for GEDI L2A granules for 2022-06-01...
Found 13 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2022-06-01 has 6355725 shots.
Performed nearest-neighbor join for 2022-06-01.
Finished GEDI L2A data extraction for months from index 12 to 17.
Total monthly GEDI L2A DataFrames collected so far: 30


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (July 2022 to December 2022, indices 18-23)
chunk_start_idx = 18
chunk_end_idx = 24

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 18 to 23...
Processing GEDI L2A data for month: 2022-07-01 to 2022-08-01
Searching for GEDI L2A granules for 2022-07-01...
Found 22 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2022-07-01 has 6061262 shots.
Performed nearest-neighbor join for 2022-07-01.
Processing GEDI L2A data for month: 2022-08-01 to 2022-09-01
Searching for GEDI L2A granules for 2022-08-01...
Found 23 GEDI L2A granules.


...

Combined GEDI L2A data for 2022-11-01 has 6333718 shots.
Performed nearest-neighbor join for 2022-11-01.
Processing GEDI L2A data for month: 2022-12-01 to 2023-01-01
Searching for GEDI L2A granules for 2022-12-01...
Found 17 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2022-12-01 has 6330719 shots.
Performed nearest-neighbor join for 2022-12-01.
Finished GEDI L2A data extraction for months from index 18 to 23.
Total monthly GEDI L2A DataFrames collected so far: 36


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (January 2023 to June 2023, indices 24-29)
chunk_start_idx = 24
chunk_end_idx = 30

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 24 to 29...
Processing GEDI L2A data for month: 2023-01-01 to 2023-02-01
Searching for GEDI L2A granules for 2023-01-01...
Found 20 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2023-01-01 has 6679699 shots.
Performed nearest-neighbor join for 2023-01-01.
Processing GEDI L2A data for month: 2023-02-01 to 2023-03-01
Searching for GEDI L2A granules for 2023-02-01...
Found 10 GEDI L2A granules.


...

Combined GEDI L2A data for 2023-02-01 has 6687315 shots.
Performed nearest-neighbor join for 2023-02-01.
Processing GEDI L2A data for month: 2023-03-01 to 2023-04-01
Searching for GEDI L2A granules for 2023-03-01...
Found 8 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2023-03-01 has 6684049 shots.
Performed nearest-neighbor join for 2023-03-01.
Processing GEDI L2A data for month: 2023-04-01 to 2023-05-01
Searching for GEDI L2A granules for 2023-04-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2023-04-01.
Skipping nearest-neighbor join for 2023-04-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2023-05-01 to 2023-06-01
Searching for GEDI L2A granules for 2023-05-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2023-05-01.
Skipping nearest-neighbor join for 2023-05-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2023-06-01 to 2023-07-01
Searching for GEDI L2A granules for 2023-06-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2023-06-01.
Skipping nearest-neighbor join for 2023-06-01 as no GEDI L2A data was collected.
Finished GEDI L2A data extraction for months from index 24 to 29.
Total monthly GEDI L2A DataFrames collected

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (July 2023 to December 2023, indices 30-35)
chunk_start_idx = 30
chunk_end_idx = 36

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 30 to 35...
Processing GEDI L2A data for month: 2023-07-01 to 2023-08-01
Searching for GEDI L2A granules for 2023-07-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2023-07-01.
Skipping nearest-neighbor join for 2023-07-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2023-08-01 to 2023-09-01
Searching for GEDI L2A granules for 2023-08-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2023-08-01.
Skipping nearest-neighbor join for 2023-08-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2023-09-01 to 2023-10-01
Searching for GEDI L2A granules for 2023-09-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2023-09-01.
Skipping nearest-neighbor join for 2023-09-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2023-10-01 to 2023-11-01
Searching for GEDI

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (January 2024 to June 2024, indices 36-41)
chunk_start_idx = 36
chunk_end_idx = 42

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 36 to 41...
Processing GEDI L2A data for month: 2024-01-01 to 2024-02-01
Searching for GEDI L2A granules for 2024-01-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2024-01-01.
Skipping nearest-neighbor join for 2024-01-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2024-02-01 to 2024-03-01
Searching for GEDI L2A granules for 2024-02-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2024-02-01.
Skipping nearest-neighbor join for 2024-02-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2024-03-01 to 2024-04-01
Searching for GEDI L2A granules for 2024-03-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2024-03-01.
Skipping nearest-neighbor join for 2024-03-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2024-04-01 to 2024-05-01
Searching for GEDI

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Combined GEDI L2A data for 2024-04-01 has 546868 shots.
Performed nearest-neighbor join for 2024-04-01.
Processing GEDI L2A data for month: 2024-05-01 to 2024-06-01
Searching for GEDI L2A granules for 2024-05-01...
Found 14 GEDI L2A granules.


...

Combined GEDI L2A data for 2024-05-01 has 4561831 shots.
Performed nearest-neighbor join for 2024-05-01.
Processing GEDI L2A data for month: 2024-06-01 to 2024-07-01
Searching for GEDI L2A granules for 2024-06-01...
Found 20 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2024-06-01 has 5318072 shots.
Performed nearest-neighbor join for 2024-06-01.
Finished GEDI L2A data extraction for months from index 36 to 41.
Total monthly GEDI L2A DataFrames collected so far: 54


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (July 2024 to December 2024, indices 42-47)
chunk_start_idx = 42
chunk_end_idx = 48

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 42 to 47...
Processing GEDI L2A data for month: 2024-07-01 to 2024-08-01
Searching for GEDI L2A granules for 2024-07-01...
Found 20 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2024-07-01 has 6688847 shots.
Performed nearest-neighbor join for 2024-07-01.
Processing GEDI L2A data for month: 2024-08-01 to 2024-09-01
Searching for GEDI L2A granules for 2024-08-01...
Found 18 GEDI L2A granules.


...

Combined GEDI L2A data for 2024-11-01 has 6672267 shots.
Performed nearest-neighbor join for 2024-11-01.
Processing GEDI L2A data for month: 2024-12-01 to 2025-01-01
Searching for GEDI L2A granules for 2024-12-01...
Found 21 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2024-12-01 has 6660176 shots.
Performed nearest-neighbor join for 2024-12-01.
Finished GEDI L2A data extraction for months from index 42 to 47.
Total monthly GEDI L2A DataFrames collected so far: 60


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (July 2024 to December 2024, indices 42-47)
chunk_start_idx = 48
chunk_end_idx = 54

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 48 to 53...
Processing GEDI L2A data for month: 2025-01-01 to 2025-02-01
Searching for GEDI L2A granules for 2025-01-01...
Found 21 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2025-01-01 has 6681582 shots.
Performed nearest-neighbor join for 2025-01-01.
Processing GEDI L2A data for month: 2025-02-01 to 2025-03-01
Searching for GEDI L2A granules for 2025-02-01...
Found 10 GEDI L2A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2A data for 2025-02-01 has 6360764 shots.
Performed nearest-neighbor join for 2025-02-01.
Processing GEDI L2A data for month: 2025-03-01 to 2025-04-01
Searching for GEDI L2A granules for 2025-03-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2025-03-01.
Skipping nearest-neighbor join for 2025-03-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2025-04-01 to 2025-05-01
Searching for GEDI L2A granules for 2025-04-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2025-04-01.
Skipping nearest-neighbor join for 2025-04-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2025-05-01 to 2025-06-01
Searching for GEDI L2A granules for 2025-05-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2025-05-01.
Skipping nearest-neighbor join for 2025-05-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2025-06-01 to 2025-07-01
Searching for GEDI L2A granules for 2025-06-01..

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Redefine the `extract_l2a` function with corrected data access paths (if not already in memory)
def extract_l2a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'rh', # 'rh' itself is a dataset with percentiles
                    'elev_lowestmode', 'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    rh_data = np.asarray(g["rh"][:], dtype=np.float64)

                    # Access rh100, rh98, rh92 from the rh_data array
                    canopy_height_rh100 = rh_data[:, 100].flatten()
                    canopy_height_rh98 = rh_data[:, 98].flatten()
                    canopy_height_rh92 = rh_data[:, 92].flatten()

                    elevation = np.asarray(g["elev_lowestmode"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(canopy_height_rh100), len(canopy_height_rh98), len(canopy_height_rh92),
                        len(elevation), len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    canopy_height_rh100 = canopy_height_rh100[:min_len]
                    canopy_height_rh98 = canopy_height_rh98[:min_len]
                    canopy_height_rh92 = canopy_height_rh92[:min_len]
                    elevation = elevation[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (canopy_height_rh100 > -9999) & (~np.isnan(canopy_height_rh100)) & \
                        (canopy_height_rh98 > -9999) & (~np.isnan(canopy_height_rh98)) & \
                        (canopy_height_rh92 > -9999) & (~np.isnan(canopy_height_rh92)) & \
                        (elevation > -9999) & (~np.isnan(elevation)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "canopy_height_rh100": float(ch100_val),
                                "canopy_height_rh98": float(ch98_val),
                                "canopy_height_rh92": float(ch92_val),
                                "elevation_lowestmode": float(elev_val)
                            }
                            for ch100_val, ch98_val, ch92_val, elev_val, lat_val, lon_val in zip(
                                canopy_height_rh100[valid_indices],
                                canopy_height_rh98[valid_indices],
                                canopy_height_rh92[valid_indices],
                                elevation[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L2A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2A extraction function redefined with corrected data access.")

# Define the chunk of months to process (July 2024 to December 2024, indices 42-47)
chunk_start_idx = 54
chunk_end_idx = 60

print(f"Starting GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2A granules
    print(f"Searching for GEDI L2A granules for {start_date_str}...")
    l2a_granules = earthaccess.search_data(
        short_name="GEDI02_A",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2a_granules)} GEDI L2A granules.")

    monthly_l2a_shots = []

    # Process L2A granules
    if l2a_granules:
        print(f"Downloading and extracting GEDI L2A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l2a_downloaded = earthaccess.download(l2a_granules[:5], out_dir_l2a_tmp)
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2a_extracted = extract_l2a(fpath)
                if not df_l2a_extracted.empty:
                    monthly_l2a_shots.append(df_l2a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2a_monthly_df = pd.DataFrame()
    if monthly_l2a_shots:
        combined_gedi_l2a_monthly_df = pd.concat(monthly_l2a_shots, ignore_index=True)
        print(f"Combined GEDI L2A data for {start_date_str} has {len(combined_gedi_l2a_monthly_df)} shots.")
    else:
        print(f"No GEDI L2A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2a = df.copy()

    # Perform nearest-neighbor join if GEDI L2A data is available
    if not combined_gedi_l2a_monthly_df.empty:
        X_gedi_l2a_monthly = combined_gedi_l2a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2A data to df_monthly_gedi_l2a
        df_monthly_gedi_l2a["GEDI_canopy_height_rh100"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh100"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh98"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh98"].values
        df_monthly_gedi_l2a["GEDI_canopy_height_rh92"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["canopy_height_rh92"].values
        df_monthly_gedi_l2a["GEDI_elevation_lowestmode"] = combined_gedi_l2a_monthly_df.iloc[idx_l2.flatten()]["elevation_lowestmode"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2a['date'] = current_month_start
    all_monthly_gedi_l2a_data.append(df_monthly_gedi_l2a)

print(f"Finished GEDI L2A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2A DataFrames collected so far: {len(all_monthly_gedi_l2a_data)}")

GEDI L2A extraction function redefined with corrected data access.
Starting GEDI L2A data extraction for months from index 54 to 59...
Processing GEDI L2A data for month: 2025-07-01 to 2025-08-01
Searching for GEDI L2A granules for 2025-07-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2025-07-01.
Skipping nearest-neighbor join for 2025-07-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2025-08-01 to 2025-09-01
Searching for GEDI L2A granules for 2025-08-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2025-08-01.
Skipping nearest-neighbor join for 2025-08-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2025-09-01 to 2025-10-01
Searching for GEDI L2A granules for 2025-09-01...
Found 0 GEDI L2A granules.
No GEDI L2A data collected for 2025-09-01.
Skipping nearest-neighbor join for 2025-09-01 as no GEDI L2A data was collected.
Processing GEDI L2A data for month: 2025-10-01 to 2025-11-01
Searching for GEDI

In [ ]:
import pandas as pd

# 1. Concatenate all DataFrames stored in the `all_monthly_gedi_l2a_data` list
final_gedi_l2a_df = pd.concat(all_monthly_gedi_l2a_data, ignore_index=True)

# 2. Define the output file path for the final GEDI L2A CSV file
output_csv_path_gedi_l2a_final = '/content/drive/MyDrive/STAGE 1/GEDI_L2A_UAE_Monthly.csv'

# 3. Save the `final_gedi_l2a_df` DataFrame to the specified CSV path
final_gedi_l2a_df.to_csv(output_csv_path_gedi_l2a_final, index=False)

# 4. Print a confirmation message
print("Monthly GEDI L2A data extraction complete.")
print(f"Final combined monthly GEDI L2A data saved to: {output_csv_path_gedi_l2a_final}")

# 5. Print the head and shape of the final DataFrame
print("Final GEDI L2A DataFrame head:")
print(final_gedi_l2a_df.head())
print(f"Final GEDI L2A DataFrame shape: {final_gedi_l2a_df.shape}")

Monthly GEDI L2A data extraction complete.
Final combined monthly GEDI L2A data saved to: /content/drive/MyDrive/STAGE 1/GEDI_L2A_UAE_Monthly.csv
Final GEDI L2A DataFrame head:
    latitude  longitude       date  GEDI_canopy_height_rh100  \
0  25.739637  54.859193 2021-01-01                       NaN   
1  25.346367  53.574829 2021-01-01                       NaN   
2  25.329268  54.673023 2021-01-01                       NaN   
3  25.257074  54.744144 2021-01-01                       NaN   
4  25.241875  54.756695 2021-01-01                       NaN   

   GEDI_canopy_height_rh98  GEDI_canopy_height_rh92  GEDI_elevation_lowestmode  
0                      NaN                      NaN                        NaN  
1                      NaN                      NaN                        NaN  
2                      NaN                      NaN                        NaN  
3                      NaN                      NaN                        NaN  
4                      NaN       

In [ ]:
final_gedi_l2a_df.describe()

,latitude,longitude,date,GEDI_canopy_height_rh100,GEDI_canopy_height_rh98,GEDI_canopy_height_rh92,GEDI_elevation_lowestmode
count,288720.000000,288720.000000,288720,128320.000000,128320.000000,128320.000000,128320.000000
mean,24.710772,54.669523,2023-02-15 13:19:59.999999488,2.998033,2.509006,1.873032,672.977944
min,22.791063,51.865832,2021-01-01 00:00:00,0.000000,0.000000,0.000000,-1398.501709
25%,24.455335,54.459660,2021-09-23 12:00:00,1.780000,1.640000,1.380000,-31.762608
50%,24.565527,54.574708,2022-12-16 12:00:00,3.100000,2.500000,1.790000,25.994003
75%,24.812508,54.756695,2024-06-08 12:00:00,3.660000,2.950000,2.120000,144.859650
max,26.195602,56.653954,2025-12-01 00:00:00,78.800003,77.449997,74.989998,9754.055664
std,0.474468,0.662975,NaN,3.267307,3.041702,2.685568,1942.272994


# GEDI L2B UAE Monthly

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# 5. Redefine the `extract_l2b` function with corrected data access paths
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 6. Initialize an empty list named `all_monthly_gedi_l2b_data`
all_monthly_gedi_l2b_data = []
print("'all_monthly_gedi_l2b_data' list initialized.")

# 7. Create a temporary directory named `/content/gedi_l2b_tmp`
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 8. Loop through the months from January 2021 to June 2021 (indices 0 to 5 of `monthly_gedi_dates`)
chunk_start_idx = 0
chunk_end_idx = 6

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 10. Remove the temporary debug directory `/content/gedi_l2b_debug`
if os.path.exists(debug_dir): shutil.rmtree(debug_dir)
print(f"Temporary debug directory '{debug_dir}' removed.")


GEDI L2B extraction function redefined with corrected data access.
'all_monthly_gedi_l2b_data' list initialized.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 0 to 5...
Processing GEDI L2B data for month: 2021-01-01 to 2021-02-01
Searching for GEDI L2B granules for 2021-01-01...
Found 11 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2021-01-01 has 4973114 shots.
Performed nearest-neighbor join for 2021-01-01.
Processing GEDI L2B data for month: 2021-02-01 to 2021-03-01
Searching for GEDI L2B granules for 2021-02-01...
Found 13 GEDI L2B granules.


...

Combined GEDI L2B data for 2021-05-01 has 4400009 shots.
Performed nearest-neighbor join for 2021-05-01.
Processing GEDI L2B data for month: 2021-06-01 to 2021-07-01
Searching for GEDI L2B granules for 2021-06-01...
Found 14 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2021-06-01 has 4856929 shots.
Performed nearest-neighbor join for 2021-06-01.
Finished GEDI L2B data extraction for months from index 0 to 5.
Total monthly GEDI L2B DataFrames collected so far: 6
Temporary debug directory '/content/gedi_l2b_debug' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 6 and chunk_end_idx = 12
chunk_start_idx = 6
chunk_end_idx = 12

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 6 to 11...
Processing GEDI L2B data for month: 2021-07-01 to 2021-08-01
Searching for GEDI L2B granules for 2021-07-01...
Found 17 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2021-07-01 has 4249654 shots.
Performed nearest-neighbor join for 2021-07-01.
Processing GEDI L2B data for month: 2021-08-01 to 2021-09-01
Searching for GEDI L2B granules for 2021-08-01...
Found 25 GEDI L2B granules.


...

Combined GEDI L2B data for 2021-11-01 has 4498527 shots.
Performed nearest-neighbor join for 2021-11-01.
Processing GEDI L2B data for month: 2021-12-01 to 2022-01-01
Searching for GEDI L2B granules for 2021-12-01...
Found 23 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2021-12-01 has 4026818 shots.
Performed nearest-neighbor join for 2021-12-01.
Finished GEDI L2B data extraction for months from index 6 to 11.
Total monthly GEDI L2B DataFrames collected so far: 12
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 12
chunk_end_idx = 18

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 12 to 17...
Processing GEDI L2B data for month: 2022-01-01 to 2022-02-01
Searching for GEDI L2B granules for 2022-01-01...
Found 18 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-01-01 has 4365631 shots.
Performed nearest-neighbor join for 2022-01-01.
Processing GEDI L2B data for month: 2022-02-01 to 2022-03-01
Searching for GEDI L2B granules for 2022-02-01...
Found 17 GEDI L2B granules.


...

Combined GEDI L2B data for 2022-05-01 has 3978572 shots.
Performed nearest-neighbor join for 2022-05-01.
Processing GEDI L2B data for month: 2022-06-01 to 2022-07-01
Searching for GEDI L2B granules for 2022-06-01...
Found 15 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-06-01 has 4005897 shots.
Performed nearest-neighbor join for 2022-06-01.
Finished GEDI L2B data extraction for months from index 12 to 17.
Total monthly GEDI L2B DataFrames collected so far: 18
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 18
chunk_end_idx = 24

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 18 to 23...
Processing GEDI L2B data for month: 2022-07-01 to 2022-08-01
Searching for GEDI L2B granules for 2022-07-01...
Found 22 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-07-01 has 3505503 shots.
Performed nearest-neighbor join for 2022-07-01.
Processing GEDI L2B data for month: 2022-08-01 to 2022-09-01
Searching for GEDI L2B granules for 2022-08-01...
Found 26 GEDI L2B granules.


...

Combined GEDI L2B data for 2022-11-01 has 4574138 shots.
Performed nearest-neighbor join for 2022-11-01.
Processing GEDI L2B data for month: 2022-12-01 to 2023-01-01
Searching for GEDI L2B granules for 2022-12-01...
Found 19 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-12-01 has 4255899 shots.
Performed nearest-neighbor join for 2022-12-01.
Finished GEDI L2B data extraction for months from index 18 to 23.
Total monthly GEDI L2B DataFrames collected so far: 24
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 12
chunk_end_idx = 18

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 12 to 17...
Processing GEDI L2B data for month: 2022-01-01 to 2022-02-01
Searching for GEDI L2B granules for 2022-01-01...
Found 18 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-01-01 has 4365631 shots.
Performed nearest-neighbor join for 2022-01-01.
Processing GEDI L2B data for month: 2022-02-01 to 2022-03-01
Searching for GEDI L2B granules for 2022-02-01...
Found 17 GEDI L2B granules.


...

Combined GEDI L2B data for 2022-05-01 has 3978572 shots.
Performed nearest-neighbor join for 2022-05-01.
Processing GEDI L2B data for month: 2022-06-01 to 2022-07-01
Searching for GEDI L2B granules for 2022-06-01...
Found 15 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-06-01 has 4005897 shots.
Performed nearest-neighbor join for 2022-06-01.
Finished GEDI L2B data extraction for months from index 12 to 17.
Total monthly GEDI L2B DataFrames collected so far: 30
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 18
chunk_end_idx = 24

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 18 to 23...
Processing GEDI L2B data for month: 2022-07-01 to 2022-08-01
Searching for GEDI L2B granules for 2022-07-01...
Found 22 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-07-01 has 3505503 shots.
Performed nearest-neighbor join for 2022-07-01.
Processing GEDI L2B data for month: 2022-08-01 to 2022-09-01
Searching for GEDI L2B granules for 2022-08-01...
Found 26 GEDI L2B granules.


...

Combined GEDI L2B data for 2022-11-01 has 4574138 shots.
Performed nearest-neighbor join for 2022-11-01.
Processing GEDI L2B data for month: 2022-12-01 to 2023-01-01
Searching for GEDI L2B granules for 2022-12-01...
Found 19 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2022-12-01 has 4255899 shots.
Performed nearest-neighbor join for 2022-12-01.
Finished GEDI L2B data extraction for months from index 18 to 23.
Total monthly GEDI L2B DataFrames collected so far: 36
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 24
chunk_end_idx = 30

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 24 to 29...
Processing GEDI L2B data for month: 2023-01-01 to 2023-02-01
Searching for GEDI L2B granules for 2023-01-01...
Found 24 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2023-01-01 has 4275283 shots.
Performed nearest-neighbor join for 2023-01-01.
Processing GEDI L2B data for month: 2023-02-01 to 2023-03-01
Searching for GEDI L2B granules for 2023-02-01...
Found 10 GEDI L2B granules.


...

Combined GEDI L2B data for 2023-02-01 has 4827427 shots.
Performed nearest-neighbor join for 2023-02-01.
Processing GEDI L2B data for month: 2023-03-01 to 2023-04-01
Searching for GEDI L2B granules for 2023-03-01...
Found 10 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2023-03-01 has 4282558 shots.
Performed nearest-neighbor join for 2023-03-01.
Processing GEDI L2B data for month: 2023-04-01 to 2023-05-01
Searching for GEDI L2B granules for 2023-04-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2023-04-01.
Skipping nearest-neighbor join for 2023-04-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2023-05-01 to 2023-06-01
Searching for GEDI L2B granules for 2023-05-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2023-05-01.
Skipping nearest-neighbor join for 2023-05-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2023-06-01 to 2023-07-01
Searching for GEDI L2B granules for 2023-06-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2023-06-01.
Skipping nearest-neighbor join for 2023-06-01 as no GEDI L2B data was collected.
Finished GEDI L2B data extraction for months from index 24 to 29.
Total monthly GEDI L2B DataFrames collected

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 30
chunk_end_idx = 36

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 30 to 35...
Processing GEDI L2B data for month: 2023-07-01 to 2023-08-01
Searching for GEDI L2B granules for 2023-07-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2023-07-01.
Skipping nearest-neighbor join for 2023-07-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2023-08-01 to 2023-09-01
Searching for GEDI L2B granules for 2023-08-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2023-08-01.
Skipping nearest-neighbor join for 2023-08-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2023-09-01 to 2023-10-01
Searching for GEDI L2B granules for 2023-09-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2023-09-01.
Skipping nearest

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 36
chunk_end_idx = 42

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 36 to 41...
Processing GEDI L2B data for month: 2024-01-01 to 2024-02-01
Searching for GEDI L2B granules for 2024-01-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2024-01-01.
Skipping nearest-neighbor join for 2024-01-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2024-02-01 to 2024-03-01
Searching for GEDI L2B granules for 2024-02-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2024-02-01.
Skipping nearest-neighbor join for 2024-02-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2024-03-01 to 2024-04-01
Searching for GEDI L2B granules for 2024-03-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2024-03-01.
Skipping nearest

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Combined GEDI L2B data for 2024-04-01 has 233973 shots.
Performed nearest-neighbor join for 2024-04-01.
Processing GEDI L2B data for month: 2024-05-01 to 2024-06-01
Searching for GEDI L2B granules for 2024-05-01...
Found 14 GEDI L2B granules.


...

Combined GEDI L2B data for 2024-05-01 has 2688386 shots.
Performed nearest-neighbor join for 2024-05-01.
Processing GEDI L2B data for month: 2024-06-01 to 2024-07-01
Searching for GEDI L2B granules for 2024-06-01...
Found 20 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2024-06-01 has 3358522 shots.
Performed nearest-neighbor join for 2024-06-01.
Finished GEDI L2B data extraction for months from index 36 to 41.
Total monthly GEDI L2B DataFrames collected so far: 54
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 42
chunk_end_idx = 48

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 42 to 47...
Processing GEDI L2B data for month: 2024-07-01 to 2024-08-01
Searching for GEDI L2B granules for 2024-07-01...
Found 22 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2024-07-01 has 3971542 shots.
Performed nearest-neighbor join for 2024-07-01.
Processing GEDI L2B data for month: 2024-08-01 to 2024-09-01
Searching for GEDI L2B granules for 2024-08-01...
Found 18 GEDI L2B granules.


...

Combined GEDI L2B data for 2024-11-01 has 5026214 shots.
Performed nearest-neighbor join for 2024-11-01.
Processing GEDI L2B data for month: 2024-12-01 to 2025-01-01
Searching for GEDI L2B granules for 2024-12-01...
Found 22 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2024-12-01 has 4603682 shots.
Performed nearest-neighbor join for 2024-12-01.
Finished GEDI L2B data extraction for months from index 42 to 47.
Total monthly GEDI L2B DataFrames collected so far: 60
Temporary download directory '/content/gedi_l2b_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 48
chunk_end_idx = 54

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 48 to 53...
Processing GEDI L2B data for month: 2025-01-01 to 2025-02-01
Searching for GEDI L2B granules for 2025-01-01...
Found 21 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2025-01-01 has 3979328 shots.
Performed nearest-neighbor join for 2025-01-01.
Processing GEDI L2B data for month: 2025-02-01 to 2025-03-01
Searching for GEDI L2B granules for 2025-02-01...
Found 11 GEDI L2B granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L2B data for 2025-02-01 has 4786752 shots.
Performed nearest-neighbor join for 2025-02-01.
Processing GEDI L2B data for month: 2025-03-01 to 2025-04-01
Searching for GEDI L2B granules for 2025-03-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2025-03-01.
Skipping nearest-neighbor join for 2025-03-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2025-04-01 to 2025-05-01
Searching for GEDI L2B granules for 2025-04-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2025-04-01.
Skipping nearest-neighbor join for 2025-04-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2025-05-01 to 2025-06-01
Searching for GEDI L2B granules for 2025-05-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2025-05-01.
Skipping nearest-neighbor join for 2025-05-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2025-06-01 to 2025-07-01
Searching for GEDI L2B granules for 2025-06-01..

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l2b` function with corrected data access paths (if not already in memory)
def extract_l2b(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group for PAI, FHD, FCOVER
                # And within the 'geolocation' group for latitude and longitude
                required_direct_keys = ['pai', 'fhd_normal', 'cover']
                required_geo_keys = ['lat_lowestmode', 'lon_lowestmode']

                if all(key in g for key in required_direct_keys) and \
                   'geolocation' in g and isinstance(g['geolocation'], h5py.Group) and \
                   all(key in g['geolocation'] for key in required_geo_keys):

                    geo_group = g['geolocation']

                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    pai = np.asarray(g["pai"][:], dtype=np.float64).flatten()
                    fhd = np.asarray(g["fhd_normal"][:], dtype=np.float64).flatten()
                    cover = np.asarray(g["cover"][:], dtype=np.float64).flatten()
                    lat = np.asarray(geo_group["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(geo_group["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(len(pai), len(fhd), len(cover), len(lat), len(lon))
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    pai = pai[:min_len]
                    fhd = fhd[:min_len]
                    cover = cover[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) for all relevant fields
                    valid_indices = (pai > -9999) & (~np.isnan(pai)) & \
                                    (fhd > -9999) & (~np.isnan(fhd)) & \
                                    (cover > -9999) & (~np.isnan(cover)) & \
                                    (lat > -9999) & (~np.isnan(lat)) & \
                                    (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "PAI_GEDI": float(pai_val),
                                "FHD_GEDI": float(fhd_val),
                                "FCOVER_GEDI": float(cover_val)
                            }
                            for pai_val, fhd_val, cover_val, lat_val, lon_val in zip(
                                pai[valid_indices],
                                fhd[valid_indices],
                                cover[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])

    except Exception as e:
        print(f"L2B extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L2B extraction function redefined with corrected data access.")

# 3. Define `out_dir_l2b_tmp` for temporary downloads and ensure its existence.
out_dir_l2b_tmp = "/content/gedi_l2b_tmp"
os.makedirs(out_dir_l2b_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l2b_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 54
chunk_end_idx = 60

print(f"Starting GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l2b_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l2b_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L2B data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L2B granules
    print(f"Searching for GEDI L2B granules for {start_date_str}...")
    l2b_granules = earthaccess.search_data(
        short_name="GEDI02_B",
        version="002",
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l2b_granules)} GEDI L2B granules.")

    monthly_l2b_shots = []

    # Process L2B granules
    if l2b_granules:
        print(f"Downloading and extracting GEDI L2B for {start_date_str}...")
        files_l2b_downloaded = earthaccess.download(l2b_granules[:5], out_dir_l2b_tmp)
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l2b_extracted = extract_l2b(fpath)
                if not df_l2b_extracted.empty:
                    monthly_l2b_shots.append(df_l2b_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l2b_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l2b_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l2b_shots:
        combined_gedi_l2b_monthly_df = pd.concat(monthly_l2b_shots, ignore_index=True)
        print(f"Combined GEDI L2B data for {start_date_str} has {len(combined_gedi_l2b_monthly_df)} shots.")
    else:
        print(f"No GEDI L2B data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l2b = df.copy()

    # Perform nearest-neighbor join if GEDI L2B data is available
    if not combined_gedi_l2b_monthly_df.empty:
        X_gedi_l2b_monthly = combined_gedi_l2b_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l2b[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l2b_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L2B data to df_monthly_gedi_l2b
        df_monthly_gedi_l2b["GEDI_PAI"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["PAI_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FHD"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FHD_GEDI"].values
        df_monthly_gedi_l2b["GEDI_FCOVER"] = combined_gedi_l2b_monthly_df.iloc[idx_l2.flatten()]["FCOVER_GEDI"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L2B data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l2b['date'] = current_month_start
    all_monthly_gedi_l2b_data.append(df_monthly_gedi_l2b)

print(f"Finished GEDI L2B data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L2B DataFrames collected so far: {len(all_monthly_gedi_l2b_data)}")

# 7. Remove the temporary debug directory `out_dir_l2b_tmp`
if os.path.exists(out_dir_l2b_tmp): shutil.rmtree(out_dir_l2b_tmp)
print(f"Temporary download directory '{out_dir_l2b_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L2B extraction function redefined with corrected data access.
Temporary directory '/content/gedi_l2b_tmp' created/ensured to exist.
Starting GEDI L2B data extraction for months from index 54 to 59...
Processing GEDI L2B data for month: 2025-07-01 to 2025-08-01
Searching for GEDI L2B granules for 2025-07-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2025-07-01.
Skipping nearest-neighbor join for 2025-07-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2025-08-01 to 2025-09-01
Searching for GEDI L2B granules for 2025-08-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2025-08-01.
Skipping nearest-neighbor join for 2025-08-01 as no GEDI L2B data was collected.
Processing GEDI L2B data for month: 2025-09-01 to 2025-10-01
Searching for GEDI L2B granules for 2025-09-01...
Found 0 GEDI L2B granules.
No GEDI L2B data collected for 2025-09-01.
Skipping nearest

In [ ]:
import pandas as pd

# 1. Concatenate all DataFrames stored in the `all_monthly_gedi_l2b_data` list
final_gedi_l2b_df = pd.concat(all_monthly_gedi_l2b_data, ignore_index=True)

# 2. Define the output file path for the final GEDI L2B CSV file
output_csv_path_gedi_l2b_final = '/content/drive/MyDrive/STAGE 1/GEDI_L2B_UAE_Monthly.csv'

# 3. Save the `final_gedi_l2b_df` DataFrame to the specified CSV path
final_gedi_l2b_df.to_csv(output_csv_path_gedi_l2b_final, index=False)

# 4. Print a confirmation message
print("Monthly GEDI L2B data extraction complete.")
print(f"Final combined monthly GEDI L2B data saved to: {output_csv_path_gedi_l2b_final}")

# 5. Print the head and shape of the final DataFrame
print("Final GEDI L2B DataFrame head:")
print(final_gedi_l2b_df.head())
print(f"Final GEDI L2B DataFrame shape: {final_gedi_l2b_df.shape}")

Monthly GEDI L2B data extraction complete.
Final combined monthly GEDI L2B data saved to: /content/drive/MyDrive/STAGE 1/GEDI_L2B_UAE_Monthly.csv
Final GEDI L2B DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER       date
0  25.739637  54.859193  0.024251  1.079394     0.012052 2021-01-01
1  25.346367  53.574829  0.007520  1.394708     0.003753 2021-01-01
2  25.329268  54.673023  0.035981  1.206098     0.017830 2021-01-01
3  25.257074  54.744144  0.003793  0.782194     0.001895 2021-01-01
4  25.241875  54.756695  0.032222  0.888996     0.015982 2021-01-01
Final GEDI L2B DataFrame shape: (288720, 6)


In [ ]:
final_gedi_l2b_df.isnull().sum()

,0
latitude,0
longitude,0
GEDI_PAI,88220
GEDI_FHD,88220
GEDI_FCOVER,88220
date,0


# GEDI L4A UAE Monthly

In [ ]:
import h5py # Ensure h5py is imported for extraction functions
import numpy as np
import os

# 1. Define the `extract_l4a` function to retrieve specified fields
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function defined to include biomass, latitude, and longitude.")

# 2. Initialize `all_monthly_gedi_l4a_data`
all_monthly_gedi_l4a_data = []
print("'all_monthly_gedi_l4a_data' list initialized.")

# 3. Create a temporary directory at `/content/gedi_l4a_tmp`
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

GEDI L4A extraction function defined to include biomass, latitude, and longitude.
'all_monthly_gedi_l4a_data' list initialized.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Define the chunk of months to process (January 2021 to June 2021, indices 0-5)
chunk_start_idx = 0
chunk_end_idx = 6

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")


Starting GEDI L4A data extraction for months from index 0 to 5...
Processing GEDI L4A data for month: 2021-01-01 to 2021-02-01
Searching for GEDI L4A granules for 2021-01-01...
Found 14 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2021-01-01 has 4143649 shots.
Performed nearest-neighbor join for 2021-01-01.
Processing GEDI L4A data for month: 2021-02-01 to 2021-03-01
Searching for GEDI L4A granules for 2021-02-01...
Found 15 GEDI L4A granules.


...

Combined GEDI L4A data for 2021-05-01 has 3175926 shots.
Performed nearest-neighbor join for 2021-05-01.
Processing GEDI L4A data for month: 2021-06-01 to 2021-07-01
Searching for GEDI L4A granules for 2021-06-01...
Found 14 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2021-06-01 has 3525215 shots.
Performed nearest-neighbor join for 2021-06-01.
Finished GEDI L4A data extraction for months from index 0 to 5.
Total monthly GEDI L4A DataFrames collected so far: 6
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# Ensure Earthdata is authenticated (already done in previous step, but for robustness)
# earthaccess.login()

# Define the chunk of months to process (July 2021 to December 2021, indices 6-11)
chunk_start_idx = 6
chunk_end_idx = 12

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")

Starting GEDI L4A data extraction for months from index 6 to 11...
Processing GEDI L4A data for month: 2021-07-01 to 2021-08-01
Searching for GEDI L4A granules for 2021-07-01...
Found 17 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2021-07-01 has 3387066 shots.
Performed nearest-neighbor join for 2021-07-01.
Processing GEDI L4A data for month: 2021-08-01 to 2021-09-01
Searching for GEDI L4A granules for 2021-08-01...
Found 25 GEDI L4A granules.


...

Combined GEDI L4A data for 2021-11-01 has 3930146 shots.
Performed nearest-neighbor join for 2021-11-01.
Processing GEDI L4A data for month: 2021-12-01 to 2022-01-01
Searching for GEDI L4A granules for 2021-12-01...
Found 23 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2021-12-01 has 2741698 shots.
Performed nearest-neighbor join for 2021-12-01.
Finished GEDI L4A data extraction for months from index 6 to 11.
Total monthly GEDI L4A DataFrames collected so far: 12
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 12
chunk_end_idx = 18

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 12 to 17...
Processing GEDI L4A data for month: 2022-01-01 to 2022-02-01
Searching for GEDI L4A granules for 2022-01-01...
Found 18 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2022-01-01 has 3498402 shots.
Performed nearest-neighbor join for 2022-01-01.
Processing GEDI L4A data for month: 2022-02-01 to 2022-03-01
Searching for GEDI L4A granules for 2022-02-01...
Found 17 GEDI L4A granules.


...

Combined GEDI L4A data for 2022-05-01 has 3491713 shots.
Performed nearest-neighbor join for 2022-05-01.
Processing GEDI L4A data for month: 2022-06-01 to 2022-07-01
Searching for GEDI L4A granules for 2022-06-01...
Found 15 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2022-06-01 has 3631384 shots.
Performed nearest-neighbor join for 2022-06-01.
Finished GEDI L4A data extraction for months from index 12 to 17.
Total monthly GEDI L4A DataFrames collected so far: 18
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 18 and chunk_end_idx = 24
chunk_start_idx = 18
chunk_end_idx = 24

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 18 to 23...
Processing GEDI L4A data for month: 2022-07-01 to 2022-08-01
Searching for GEDI L4A granules for 2022-07-01...
Found 22 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2022-07-01 has 3203958 shots.
Performed nearest-neighbor join for 2022-07-01.
Processing GEDI L4A data for month: 2022-08-01 to 2022-09-01
Searching for GEDI L4A granules for 2022-08-01...
Found 26 GEDI L4A granules.


...

Combined GEDI L4A data for 2022-11-01 has 3693139 shots.
Performed nearest-neighbor join for 2022-11-01.
Processing GEDI L4A data for month: 2022-12-01 to 2023-01-01
Searching for GEDI L4A granules for 2022-12-01...
Found 19 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2022-12-01 has 3659711 shots.
Performed nearest-neighbor join for 2022-12-01.
Finished GEDI L4A data extraction for months from index 18 to 23.
Total monthly GEDI L4A DataFrames collected so far: 24
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 24
chunk_end_idx = 30

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 24 to 29...
Processing GEDI L4A data for month: 2023-01-01 to 2023-02-01
Searching for GEDI L4A granules for 2023-01-01...
Found 24 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2023-01-01 has 2856442 shots.
Performed nearest-neighbor join for 2023-01-01.
Processing GEDI L4A data for month: 2023-02-01 to 2023-03-01
Searching for GEDI L4A granules for 2023-02-01...
Found 10 GEDI L4A granules.


...

Combined GEDI L4A data for 2023-02-01 has 4124763 shots.
Performed nearest-neighbor join for 2023-02-01.
Processing GEDI L4A data for month: 2023-03-01 to 2023-04-01
Searching for GEDI L4A granules for 2023-03-01...
Found 10 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2023-03-01 has 3657053 shots.
Performed nearest-neighbor join for 2023-03-01.
Processing GEDI L4A data for month: 2023-04-01 to 2023-05-01
Searching for GEDI L4A granules for 2023-04-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2023-04-01.
Skipping nearest-neighbor join for 2023-04-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2023-05-01 to 2023-06-01
Searching for GEDI L4A granules for 2023-05-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2023-05-01.
Skipping nearest-neighbor join for 2023-05-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2023-06-01 to 2023-07-01
Searching for GEDI L4A granules for 2023-06-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2023-06-01.
Skipping nearest-neighbor join for 2023-06-01 as no GEDI L4A data was collected.
Finished GEDI L4A data extraction for months from index 24 to 29.
Total monthly GEDI L4A DataFrames collected

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 30
chunk_end_idx = 36

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 30 to 35...
Processing GEDI L4A data for month: 2023-07-01 to 2023-08-01
Searching for GEDI L4A granules for 2023-07-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2023-07-01.
Skipping nearest-neighbor join for 2023-07-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2023-08-01 to 2023-09-01
Searching for GEDI L4A granules for 2023-08-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2023-08-01.
Skipping nearest-neighbor join for 2023-08-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2023-09-01 to 2023-10-01
Searching for GEDI L4A granules for 2023-09-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2023-09-01.

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 36
chunk_end_idx = 42

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")

Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 36 to 41...
Processing GEDI L4A data for month: 2024-01-01 to 2024-02-01
Searching for GEDI L4A granules for 2024-01-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2024-01-01.
Skipping nearest-neighbor join for 2024-01-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2024-02-01 to 2024-03-01
Searching for GEDI L4A granules for 2024-02-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2024-02-01.
Skipping nearest-neighbor join for 2024-02-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2024-03-01 to 2024-04-01
Searching for GEDI L4A granules for 2024-03-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2024-03-01.

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Combined GEDI L4A data for 2024-04-01 has 204473 shots.
Performed nearest-neighbor join for 2024-04-01.
Processing GEDI L4A data for month: 2024-05-01 to 2024-06-01
Searching for GEDI L4A granules for 2024-05-01...
Found 14 GEDI L4A granules.


...

Combined GEDI L4A data for 2024-05-01 has 2147462 shots.
Performed nearest-neighbor join for 2024-05-01.
Processing GEDI L4A data for month: 2024-06-01 to 2024-07-01
Searching for GEDI L4A granules for 2024-06-01...
Found 20 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2024-06-01 has 3092183 shots.
Performed nearest-neighbor join for 2024-06-01.
Finished GEDI L4A data extraction for months from index 36 to 41.
Total monthly GEDI L4A DataFrames collected so far: 42
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 42
chunk_end_idx = 48

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 42 to 47...
Processing GEDI L4A data for month: 2024-07-01 to 2024-08-01
Searching for GEDI L4A granules for 2024-07-01...
Found 22 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2024-07-01 has 3282255 shots.
Performed nearest-neighbor join for 2024-07-01.
Processing GEDI L4A data for month: 2024-08-01 to 2024-09-01
Searching for GEDI L4A granules for 2024-08-01...
Found 18 GEDI L4A granules.


...

Combined GEDI L4A data for 2024-11-01 has 3598995 shots.
Performed nearest-neighbor join for 2024-11-01.
Processing GEDI L4A data for month: 2024-12-01 to 2025-01-01
Searching for GEDI L4A granules for 2024-12-01...
Found 22 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2024-12-01 has 3597807 shots.
Performed nearest-neighbor join for 2024-12-01.
Finished GEDI L4A data extraction for months from index 42 to 47.
Total monthly GEDI L4A DataFrames collected so far: 48
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 48
chunk_end_idx = 54

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 48 to 53...
Processing GEDI L4A data for month: 2025-01-01 to 2025-02-01
Searching for GEDI L4A granules for 2025-01-01...
Found 21 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2025-01-01 has 3157925 shots.
Performed nearest-neighbor join for 2025-01-01.
Processing GEDI L4A data for month: 2025-02-01 to 2025-03-01
Searching for GEDI L4A granules for 2025-02-01...
Found 14 GEDI L4A granules.


...

Combined GEDI L4A data for 2025-02-01 has 4186424 shots.
Performed nearest-neighbor join for 2025-02-01.
Processing GEDI L4A data for month: 2025-03-01 to 2025-04-01
Searching for GEDI L4A granules for 2025-03-01...
Found 11 GEDI L4A granules.


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

Combined GEDI L4A data for 2025-03-01 has 3038115 shots.
Performed nearest-neighbor join for 2025-03-01.
Processing GEDI L4A data for month: 2025-04-01 to 2025-05-01
Searching for GEDI L4A granules for 2025-04-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2025-04-01.
Skipping nearest-neighbor join for 2025-04-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2025-05-01 to 2025-06-01
Searching for GEDI L4A granules for 2025-05-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2025-05-01.
Skipping nearest-neighbor join for 2025-05-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2025-06-01 to 2025-07-01
Searching for GEDI L4A granules for 2025-06-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2025-06-01.
Skipping nearest-neighbor join for 2025-06-01 as no GEDI L4A data was collected.
Finished GEDI L4A data extraction for months from index 48 to 53.
Total monthly GEDI L4A DataFrames collected

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx = 12 and chunk_end_idx = 18
chunk_start_idx = 54
chunk_end_idx = 60

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 54 to 59...
Processing GEDI L4A data for month: 2025-07-01 to 2025-08-01
Searching for GEDI L4A granules for 2025-07-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2025-07-01.
Skipping nearest-neighbor join for 2025-07-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2025-08-01 to 2025-09-01
Searching for GEDI L4A granules for 2025-08-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2025-08-01.
Skipping nearest-neighbor join for 2025-08-01 as no GEDI L4A data was collected.
Processing GEDI L4A data for month: 2025-09-01 to 2025-10-01
Searching for GEDI L4A granules for 2025-09-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2025-09-01.

In [ ]:
import pandas as pd
import earthaccess
import os
from sklearn.neighbors import NearestNeighbors
import numpy as np
import h5py
import shutil

# 1. Re-authenticate with Earthdata
print("Re-authenticating with Earthdata...")
earthaccess.login()
print("Earthdata re-authentication complete.")

# Redefine the `extract_l4a` function with corrected data access paths (if not already in memory)
def extract_l4a(path):
    rows = []
    try:
        with h5py.File(path, "r") as f:
            for beam in f.keys():
                if not beam.startswith("BEAM"):
                    continue
                g = f[beam]

                # Check for required datasets directly under the beam group
                required_keys = [
                    'agbd',
                    'lat_lowestmode', 'lon_lowestmode'
                ]

                if all(key in g for key in required_keys):
                    # Explicitly convert to numpy array and flatten to ensure 1D float arrays
                    agbd = np.asarray(g["agbd"][:], dtype=np.float64).flatten()
                    lat = np.asarray(g["lat_lowestmode"][:], dtype=np.float64).flatten()
                    lon = np.asarray(g["lon_lowestmode"][:], dtype=np.float64).flatten()

                    # Ensure all arrays have the same length before filtering
                    min_len = min(
                        len(agbd),
                        len(lat), len(lon)
                    )
                    if min_len == 0:
                        continue # Skip this beam if no data after flattening/length check

                    agbd = agbd[:min_len]
                    lat = lat[:min_len]
                    lon = lon[:min_len]

                    # Filter out fill values (-9999 or NaN) using vectorized operations
                    valid_indices = \
                        (agbd > -9999) & (~np.isnan(agbd)) & \
                        (lat > -9999) & (~np.isnan(lat)) & \
                        (lon > -9999) & (~np.isnan(lon))

                    if valid_indices.any():
                        rows.extend([
                            {
                                "latitude": float(lat_val),
                                "longitude": float(lon_val),
                                "biomass_Mg_ha": float(agbd_val)
                            }
                            for agbd_val, lat_val, lon_val in zip(
                                agbd[valid_indices],
                                lat[valid_indices],
                                lon[valid_indices]
                            )
                        ])
    except Exception as e:
        print(f"L4A extraction error for file {path}: {e}")
    return pd.DataFrame(rows)
print("GEDI L4A extraction function redefined to include biomass, latitude, and longitude.")

# 3. Define `out_dir_l4a_tmp` for temporary downloads and ensure its existence.
out_dir_l4a_tmp = "/content/gedi_l4a_tmp"
os.makedirs(out_dir_l4a_tmp, exist_ok=True)
print(f"Temporary directory '{out_dir_l4a_tmp}' created/ensured to exist.")

# 4. Set chunk_start_idx and chunk_end_idx for the final month (January 2026)
chunk_start_idx = 60
chunk_end_idx = min(chunk_start_idx + 1, len(monthly_gedi_dates)) # Process only one month for the final chunk

print(f"Starting GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}...")

# `all_monthly_gedi_l4a_data` was initialized in a previous step, ensuring continuity.
# If the kernel reset, uncomment the following line to re-initialize:
# all_monthly_gedi_l4a_data = []

for i in range(chunk_start_idx, chunk_end_idx):
    current_month_start = monthly_gedi_dates[i]
    # Calculate next month's start for the end date filter
    if i + 1 < len(monthly_gedi_dates):
        next_month_start = monthly_gedi_dates[i+1]
    else:
        # For the very last month, ensure the period covers the entire month
        next_month_start = current_month_start + pd.DateOffset(months=1)

    start_date_str = current_month_start.strftime('%Y-%m-%d')
    end_date_str = next_month_start.strftime('%Y-%m-%d')

    print(f"Processing GEDI L4A data for month: {start_date_str} to {end_date_str}")

    temporal = (start_date_str, end_date_str)

    # Search for GEDI L4A granules
    print(f"Searching for GEDI L4A granules for {start_date_str}...")
    l4a_granules = earthaccess.search_data(
        concept_id="C2237824918-ORNL_CLOUD", # Using Concept ID as specified in initial notebook
        bounding_box=bbox,
        temporal=temporal
    )
    print(f"Found {len(l4a_granules)} GEDI L4A granules.")

    monthly_l4a_shots = []

    # Process L4A granules
    if l4a_granules:
        print(f"Downloading and extracting GEDI L4A for {start_date_str}...")
        # Download a reasonable number of granules to avoid excessive processing/memory
        files_l4a_downloaded = earthaccess.download(l4a_granules[:5], out_dir_l4a_tmp)
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Check if the downloaded file exists before processing
                df_l4a_extracted = extract_l4a(fpath)
                if not df_l4a_extracted.empty:
                    monthly_l4a_shots.append(df_l4a_extracted)
            else:
                print(f"Warning: Downloaded file not found at {fpath}. Skipping extraction.")
        # Clean up *only* the downloaded GEDI files after processing
        for fpath in files_l4a_downloaded:
            if os.path.exists(fpath): # Ensure the file exists before attempting to remove
                os.remove(fpath)

    combined_gedi_l4a_monthly_df = pd.DataFrame() # Initialize for cases with no shots
    if monthly_l4a_shots:
        combined_gedi_l4a_monthly_df = pd.concat(monthly_l4a_shots, ignore_index=True)
        print(f"Combined GEDI L4A data for {start_date_str} has {len(combined_gedi_l4a_monthly_df)} shots.")
    else:
        print(f"No GEDI L4A data collected for {start_date_str}.")

    # Create a copy of the base df for this month's data
    df_monthly_gedi_l4a = df.copy()

    # Perform nearest-neighbor join if GEDI L4A data is available
    if not combined_gedi_l4a_monthly_df.empty:
        X_gedi_l4a_monthly = combined_gedi_l4a_monthly_df[["latitude", "longitude"]].to_numpy()
        X_pts = df_monthly_gedi_l4a[["latitude", "longitude"]].to_numpy()

        nn = NearestNeighbors(n_neighbors=1)
        nn.fit(X_gedi_l4a_monthly)
        dist_l2, idx_l2 = nn.kneighbors(X_pts)

        # Add L4A data to df_monthly_gedi_l4a
        df_monthly_gedi_l4a["GEDI_biomass_Mg_ha"] = combined_gedi_l4a_monthly_df.iloc[idx_l2.flatten()]["biomass_Mg_ha"].values
        print(f"Performed nearest-neighbor join for {start_date_str}.")
    else:
        print(f"Skipping nearest-neighbor join for {start_date_str} as no GEDI L4A data was collected.")

    # Add date column and append to the main list
    df_monthly_gedi_l4a['date'] = current_month_start
    all_monthly_gedi_l4a_data.append(df_monthly_gedi_l4a)

print(f"Finished GEDI L4A data extraction for months from index {chunk_start_idx} to {chunk_end_idx - 1}.")
print(f"Total monthly GEDI L4A DataFrames collected so far: {len(all_monthly_gedi_l4a_data)}")

# Remove the temporary download directory `out_dir_l4a_tmp`
if os.path.exists(out_dir_l4a_tmp): shutil.rmtree(out_dir_l4a_tmp)
print(f"Temporary download directory '{out_dir_l4a_tmp}' removed.")


Re-authenticating with Earthdata...
Earthdata re-authentication complete.
GEDI L4A extraction function redefined to include biomass, latitude, and longitude.
Temporary directory '/content/gedi_l4a_tmp' created/ensured to exist.
Starting GEDI L4A data extraction for months from index 60 to 60...
Processing GEDI L4A data for month: 2026-01-01 to 2026-02-01
Searching for GEDI L4A granules for 2026-01-01...
Found 0 GEDI L4A granules.
No GEDI L4A data collected for 2026-01-01.
Skipping nearest-neighbor join for 2026-01-01 as no GEDI L4A data was collected.
Finished GEDI L4A data extraction for months from index 60 to 60.
Total monthly GEDI L4A DataFrames collected so far: 61
Temporary download directory '/content/gedi_l4a_tmp' removed.


In [ ]:
import pandas as pd

# 1. Concatenate all DataFrames stored in the `all_monthly_gedi_l4a_data` list
final_gedi_l4a_df = pd.concat(all_monthly_gedi_l4a_data, ignore_index=True)

# 2. Define the output file path for the final GEDI L4A CSV file
output_csv_path_gedi_l4a_final = '/content/drive/MyDrive/STAGE 1/GEDI_L4A_UAE_Monthly.csv'

# 3. Save the `final_gedi_l4a_df` DataFrame to the specified CSV path
final_gedi_l4a_df.to_csv(output_csv_path_gedi_l4a_final, index=False)

# 4. Print a confirmation message
print("Monthly GEDI L4A data extraction complete.")
print(f"Final combined monthly GEDI L4A data saved to: {output_csv_path_gedi_l4a_final}")

# 5. Print the head and shape of the final DataFrame
print("Final GEDI L4A DataFrame head:")
print(final_gedi_l4a_df.head())
print(f"Final GEDI L4A DataFrame shape: {final_gedi_l4a_df.shape}")

Monthly GEDI L4A data extraction complete.
Final combined monthly GEDI L4A data saved to: /content/drive/MyDrive/STAGE 1/GEDI_L4A_UAE_Monthly.csv
Final GEDI L4A DataFrame head:
    latitude  longitude  GEDI_biomass_Mg_ha       date
0  25.739637  54.859193            0.709250 2021-01-01
1  25.346367  53.574829            0.624369 2021-01-01
2  25.329268  54.673023            0.709250 2021-01-01
3  25.257074  54.744144            0.709250 2021-01-01
4  25.241875  54.756695            0.709250 2021-01-01
Final GEDI L4A DataFrame shape: (244610, 4)


In [ ]:
final_gedi_l4a_df.isnull().sum()

,0
latitude,0
longitude,0
date,0
GEDI_canopy_height_rh100,160400
GEDI_canopy_height_rh98,160400
GEDI_canopy_height_rh92,160400
GEDI_elevation_lowestmode,160400


# GEDI L2A L2B L4A Combining

In [ ]:
import pandas as pd

# Load GEDI L2A data
gedi_l2a_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_L2A_UAE_Monthly.csv')
gedi_l2a_df['date'] = pd.to_datetime(gedi_l2a_df['date'])

# Load GEDI L2B data
gedi_l2b_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_L2B_UAE_Monthly.csv')
gedi_l2b_df['date'] = pd.to_datetime(gedi_l2b_df['date'])

# Load GEDI L4A data
gedi_l4a_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_L4A_UAE_Monthly.csv')
gedi_l4a_df['date'] = pd.to_datetime(gedi_l4a_df['date'])

print("GEDI L2A DataFrame head:")
print(gedi_l2a_df.head())
print("\nGEDI L2B DataFrame head:")
print(gedi_l2b_df.head())
print("\nGEDI L4A DataFrame head:")
print(gedi_l4a_df.head())

GEDI L2A DataFrame head:
    latitude  longitude       date  GEDI_canopy_height_rh100  \
0  25.739637  54.859193 2021-01-01                       NaN   
1  25.346367  53.574829 2021-01-01                       NaN   
2  25.329268  54.673023 2021-01-01                       NaN   
3  25.257074  54.744144 2021-01-01                       NaN   
4  25.241875  54.756695 2021-01-01                       NaN   

   GEDI_canopy_height_rh98  GEDI_canopy_height_rh92  GEDI_elevation_lowestmode  
0                      NaN                      NaN                        NaN  
1                      NaN                      NaN                        NaN  
2                      NaN                      NaN                        NaN  
3                      NaN                      NaN                        NaN  
4                      NaN                      NaN                        NaN  

GEDI L2B DataFrame head:
    latitude  longitude  GEDI_PAI  GEDI_FHD  GEDI_FCOVER       date
0  25.7396

In [ ]:
import pandas as pd

# Load GEDI L2A data
gedi_l2a_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_L2A_UAE_Monthly.csv')
gedi_l2a_df['date'] = pd.to_datetime(gedi_l2a_df['date'])

# Load GEDI L2B data
gedi_l2b_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_L2B_UAE_Monthly.csv')
gedi_l2b_df['date'] = pd.to_datetime(gedi_l2b_df['date'])

# Load GEDI L4A data
gedi_l4a_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_L4A_UAE_Monthly.csv')
gedi_l4a_df['date'] = pd.to_datetime(gedi_l4a_df['date'])

# Aggregate GEDI L2A, L2B, and L4A DataFrames to ensure unique lat/lon/date combinations
# Taking the mean of GEDI-specific columns for aggregation
gedi_l2a_agg = gedi_l2a_df.groupby(['latitude', 'longitude', 'date'], as_index=False).mean()
gedi_l2b_agg = gedi_l2b_df.groupby(['latitude', 'longitude', 'date'], as_index=False).mean()
gedi_l4a_agg = gedi_l4a_df.groupby(['latitude', 'longitude', 'date'], as_index=False).mean()

# Create a master DataFrame with all unique coordinate-date combinations
# `df` contains the unique latitude/longitude pairs (4010 rows) from previous kernel state
# `monthly_gedi_dates` contains the monthly date ranges (61 months) from previous kernel state

# Convert monthly_gedi_dates list to a DataFrame for merging
dates_df = pd.DataFrame({'date': monthly_gedi_dates})

# Create the master DataFrame by taking a Cartesian product of df and dates_df
# This ensures every coordinate exists for every month
master_df = pd.merge(
    df.assign(key=1),
    dates_df.assign(key=1),
    on='key'
).drop('key', axis=1)

# Ensure master_df 'date' column is datetime
master_df['date'] = pd.to_datetime(master_df['date'])

# Perform left merges to add aggregated GEDI data
# Merge L2A data
merged_gedi_df = pd.merge(
    master_df,
    gedi_l2a_agg,
    on=['latitude', 'longitude', 'date'],
    how='left'
)

# Merge L2B data onto the result
merged_gedi_df = pd.merge(
    merged_gedi_df,
    gedi_l2b_agg,
    on=['latitude', 'longitude', 'date'],
    how='left'
)

# Merge L4A data onto the result
final_gedi_merged_df = pd.merge(
    merged_gedi_df,
    gedi_l4a_agg,
    on=['latitude', 'longitude', 'date'],
    how='left'
)

# Define the output path for the final merged GEDI data
output_csv_path_gedi_new = '/content/drive/MyDrive/STAGE 1/GEDI_NEW_UAE_Monthly.csv'

# Save the final merged DataFrame to a CSV file
final_gedi_merged_df.to_csv(output_csv_path_gedi_new, index=False)

print(f"Final combined GEDI data saved to: {output_csv_path_gedi_new}")
print("\nFinal Merged GEDI DataFrame head:")
print(final_gedi_merged_df.head())
print(f"\nFinal Merged GEDI DataFrame shape: {final_gedi_merged_df.shape}")

Final combined GEDI data saved to: /content/drive/MyDrive/STAGE 1/GEDI_NEW_UAE_Monthly.csv

Final Merged GEDI DataFrame head:
    latitude  longitude       date  GEDI_canopy_height_rh100  \
0  25.739637  54.859193 2021-01-01                       NaN   
1  25.739637  54.859193 2021-02-01                       NaN   
2  25.739637  54.859193 2021-03-01                       NaN   
3  25.739637  54.859193 2021-04-01                       NaN   
4  25.739637  54.859193 2021-05-01                       NaN   

   GEDI_canopy_height_rh98  GEDI_canopy_height_rh92  \
0                      NaN                      NaN   
1                      NaN                      NaN   
2                      NaN                      NaN   
3                      NaN                      NaN   
4                      NaN                      NaN   

   GEDI_elevation_lowestmode  GEDI_PAI  GEDI_FHD  GEDI_FCOVER  \
0                        NaN  0.024251  1.079394     0.012052   
1                        NaN

# S1 S2 GEDI New Combining

In [ ]:
import pandas as pd

s1_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/S1_UAE_Monthly.csv')
s1_df['date'] = pd.to_datetime(s1_df['date'])

s2_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/S2_UAE_Monthly.csv')
s2_df['date'] = pd.to_datetime(s2_df['date'])

gedi_df = pd.read_csv('/content/drive/MyDrive/STAGE 1/GEDI_NEW_UAE_Monthly.csv')
gedi_df['date'] = pd.to_datetime(gedi_df['date'])

s1_df = s1_df.sort_values(by=['latitude', 'longitude', 'date']).reset_index(drop=True)
s2_df = s2_df.sort_values(by=['latitude', 'longitude', 'date']).reset_index(drop=True)
gedi_df = gedi_df.sort_values(by=['latitude', 'longitude', 'date']).reset_index(drop=True)


In [ ]:
# Drop common columns from s2_df and gedi_df to avoid duplication during concatenation
s2_df_unique = s2_df.drop(columns=['latitude', 'longitude', 'date'])
gedi_df_unique = gedi_df.drop(columns=['latitude', 'longitude', 'date'])

print(s2_df_unique.shape)
print(gedi_df_unique.shape)

(244610, 23)
(244610, 8)


In [ ]:
final_merged_side_by_side_df = pd.concat([s1_df, s2_df_unique, gedi_df_unique], axis=1)
output_csv_path_merged_sbs = '/content/drive/MyDrive/STAGE 1/S1_S2_GEDI_New_UAE_Monthly.csv'
final_merged_side_by_side_df.to_csv(output_csv_path_merged_sbs, index=False)

print(f"Side-by-side merged data saved to: {output_csv_path_merged_sbs}")
print("Final Merged (Side-by-Side) DataFrame head:")
print(final_merged_side_by_side_df.head())
print(f"Final Merged (Side-by-Side) DataFrame shape: {final_merged_side_by_side_df.shape}")

Side-by-side merged data saved to: /content/drive/MyDrive/STAGE 1/S1_S2_GEDI_New_UAE_Monthly.csv
Final Merged (Side-by-Side) DataFrame head:
    latitude  longitude  VV_backscatter  VH_backscatter  VV_coherence  \
0  22.791063  54.246296      -24.314251      -31.478721      0.003703   
1  22.791063  54.246296      -25.507535      -30.963398      0.002819   
2  22.791063  54.246296      -25.626493      -31.186509      0.002737   
3  22.791063  54.246296      -23.189345      -31.998171      0.004816   
4  22.791063  54.246296      -22.832819      -25.969246      0.005209   

   VH_coherence       date     B1      B2      B3  ...  \
0      0.000711 2021-01-01  755.0  1004.5  1856.5  ...   
1      0.000801 2021-02-01  743.0  1055.0  1854.0  ...   
2      0.000761 2021-03-01  693.5  1009.0  1882.0  ...   
3      0.000642 2021-04-01  733.0  1026.0  1736.0  ...   
4      0.002530 2021-05-01  756.5  1032.0  1804.0  ...   

   Highlight_Optimized_Natural_Color  True_color  GEDI_canopy_height_rh